# Offline Evaluation for Recommender Systems, Search, and Personalization
## A Mastery-Level Study Guide for Staff Engineers

---

> **Audience:** Staff Engineers with deep backend and ML systems experience who want to develop genuine expertise — not working knowledge — in offline evaluation methodology.
>
> **Philosophy:** Every claim is grounded in math or a citable result. Every technique comes with its failure conditions. Theory is always connected to practice.

---

## Table of Contents

| Level | Topic |
|-------|-------|
| 1 | [Foundations — Why Offline Eval Is Hard](#level-1) |
| 2 | [Core Metrics — Derivations and Tradeoffs](#level-2) |
| 3 | [Debiasing Techniques](#level-3) |
| 4 | [Offline-Online Correlation and Metric Design](#level-4) |
| 5 | [Industry Practice and Canonical Papers](#level-5) |
| B | [Recommended Reading — Textbooks and Articles](#appendix-b) |
| C | [Search vs. Recommendation — Where the Framework Applies](#appendix-c) |

---

<a id='level-1'></a>
# LEVEL 1: FOUNDATIONS — WHY OFFLINE EVAL IS HARD

---

## 1.1 IID Holdout Evaluation vs. Evaluation Under a Logged Policy

### The Standard ML Contract

In standard supervised learning, the evaluation contract is clean: you assume that your training set and test set are drawn i.i.d. from the same data-generating distribution $\mathcal{D}$. If your model achieves low loss on a held-out test set, you have a statistically valid estimate of its expected loss under $\mathcal{D}$.

Formally, if $(x_i, y_i) \overset{\text{i.i.d.}}{\sim} \mathcal{D}$ for $i = 1, \ldots, n$, then for any bounded loss function $\ell$:

$$\hat{R}(f) = \frac{1}{n} \sum_{i=1}^{n} \ell(f(x_i), y_i) \xrightarrow{p} \mathbb{E}_{(x,y) \sim \mathcal{D}}[\ell(f(x), y)]$$

This is just the law of large numbers. No further assumptions required.

### Why Recommender Systems Break This Contract

In a recommender system, the observed data is not drawn i.i.d. from $\mathcal{D}$. It is drawn from an interaction process controlled by a **logging policy** $\pi_0$. The logging policy is the deployed system that decided which items to show to which users at which positions.

Let:
- $u \in \mathcal{U}$: a user
- $i \in \mathcal{I}$: an item
- $\mathbf{a} = (a_1, \ldots, a_K)$: an ordered list (slate) of $K$ items shown to the user
- $r(u, i)$: the **true** relevance of item $i$ for user $u$ (a random variable we cannot observe directly)
- $o(u, i, \mathbf{a})$: a binary observation indicator — whether we **get to see** the relevance signal at all
- $c(u, i)$: the observed outcome (e.g., click) — which we only observe when $o = 1$

The data-generating process is:

1. User $u$ arrives with context $x_u$
2. Logging policy selects a slate: $\mathbf{a} \sim \pi_0(\cdot | x_u)$
3. User interacts; we observe $c(u, i)$ **only for items in $\mathbf{a}$**
4. For all items $i \notin \mathbf{a}$: $r(u, i)$ is **permanently unobserved**

Now suppose you want to evaluate a **new policy** $\pi_e$ (the evaluation target) — a new ranker that would produce a different slate. The quantity you want is:

$$V(\pi_e) = \mathbb{E}_{u \sim \mathcal{U}}\left[\mathbb{E}_{\mathbf{a} \sim \pi_e(\cdot|x_u)}\left[\sum_{i \in \mathbf{a}} r(u, i) \cdot w(i, \mathbf{a})\right]\right]$$

where $w(i, \mathbf{a})$ is a position-weighted relevance contribution. But you **cannot compute this** from logged data, because:

1. $r(u, i)$ is unobserved for most $(u, i)$ pairs
2. Even for observed pairs, the observation probability depends on $\pi_0$, not $\pi_e$

**The naive approach:** Use the data as if it were i.i.d. — treat all unclicked items as irrelevant, and compute NDCG or MAP against this dataset. This is **deeply wrong** for reasons elaborated in Section 1.2–1.4.

### The Bandit Feedback Setting

Recommendation evaluation is structurally a **bandit problem**. You observe reward only for the arm (item/slate) that the logging policy pulled. This is fundamentally different from full-information feedback where you would observe the counterfactual reward for all possible arms.

The gap between these two settings is not just statistical — it is **epistemological**: no amount of additional data from $\pi_0$ will tell you how a user would have responded to an item that $\pi_0$ never showed them.

## 1.2 The Three Core Biases in Interaction Data

### 1.2.1 Position Bias

**Definition:** Users are more likely to examine and click items shown at higher positions, independent of item quality. The click probability for item $i$ at position $k$ is a product of examination probability and relevance:

$$P(\text{click} \mid i, k) = P(\text{examine} \mid k) \cdot P(\text{click} \mid i, \text{examine})$$

Let $\theta_k = P(\text{examine} \mid k)$ be the **examination probability** at rank $k$. Empirically (Joachims et al., 2005; Craswell et al., 2008):

$$\theta_k \approx \frac{1}{k^\gamma}$$

for some $\gamma > 0$ that depends on the UI and query type. On web search, $\gamma \approx 0.9\text{–}1.5$. On mobile, positional effects are even more extreme.

**Effect on NDCG:** Consider two rankings $\sigma_1$ and $\sigma_2$. Under position bias, item $i$ at position 1 will accumulate more clicks than the same item at position 5, purely because $\theta_1 \gg \theta_5$. If you now train a model on this data and use NDCG to evaluate:

$$\text{DCG}@K = \sum_{k=1}^{K} \frac{\text{rel}_k}{\log_2(k+1)}$$

the measured $\text{rel}_k$ is not the true relevance — it is the **click rate**, which is confounded with $\theta_k$. Specifically:

$$\hat{\text{rel}}_k = \mathbb{E}[\text{click} \mid k] = \theta_k \cdot \text{rel}(i_k)$$

A model that systematically places high-examination items at top positions will have inflated observed NDCG, even if those items are not intrinsically more relevant. The NDCG discount and the examination probability pull in the **same** direction, compounding the bias:

$$\mathbb{E}[\text{biased NDCG}] = \sum_{k=1}^{K} \frac{\theta_k \cdot \text{rel}(i_k)}{\log_2(k+1)}$$

vs. the true NDCG:

$$\text{true NDCG} = \frac{1}{\text{IDCG}} \sum_{k=1}^{K} \frac{\text{rel}(i_k)}{\log_2(k+1)}$$

These only agree when $\theta_k = c$ (constant examination, which never holds in practice).

### 1.2.2 Popularity Bias

**Definition:** Item frequency in training data follows a long-tail (Zipfian) distribution. Popular items are shown more often, clicked more often, and therefore accumulate more signal. Rare items remain data-starved.

Formally, let $n_i$ be the number of times item $i$ appears in the logged dataset. In most production systems:

$$n_i \propto \text{rank}(i)^{-\alpha}, \quad \alpha \in [0.8, 1.5]$$

This Zipf distribution means that the top 1% of items may account for 50%+ of all interactions.

**Effect on Coverage:** Coverage measures how much of the item catalog a system utilizes:

$$\text{Coverage} = \frac{|\{i : i \text{ appears in at least one recommendation list}\}|}{|\mathcal{I}|}$$

A popularity-biased system trained on logged data will learn to recommend popular items because they have more training signal, even when long-tail items may be more relevant for specific users. Evaluated on the same biased log, this system appears to have high Recall@K because the test set is also biased toward popular items.

**Effect on Recall:** If your test set is a held-out slice of your logged interactions, popular items dominate the set of held-out relevant items. A system that recommends only the top-100 most popular items will have artificially high Recall@K because those items appear in most users' held-out interactions — not because the system has learned user preferences.

**Mathematical formulation:** Let $\mathcal{R}(u)$ be user $u$'s true relevant items and $\hat{\mathcal{R}}(u)$ be their observed relevant items in the log. Because $P(i \in \hat{\mathcal{R}}(u)) \propto n_i$, we have:

$$\mathbb{E}[\text{Recall@K observed}] = \frac{\mathbb{E}[|\hat{S}(u) \cap \hat{\mathcal{R}}(u)|]}{\mathbb{E}[|\hat{\mathcal{R}}(u)|]}$$

where $\hat{S}(u)$ is the top-K recommended items. Popular items are overrepresented in both numerator and denominator, creating a false impression of recall.

### 1.2.3 Selection Bias / Exposure Bias

**Definition:** You can only observe relevance for items that were shown to the user. This is the **missing-not-at-random (MNAR)** problem: missingness of feedback is correlated with the value of the missing data.

Formally, let $M_{u,i} \in \{0, 1\}$ be the indicator that item $i$ was shown to user $u$. The observed data is:

$$\mathcal{D}_{\text{obs}} = \{(u, i, r_{u,i}) : M_{u,i} = 1\}$$

The missing data problem is:

$$P(M_{u,i} = 1) = P(i \in \mathbf{a} \mid u, \pi_0)$$

which depends on $\pi_0$'s estimate of relevance — creating a circular dependency: items estimated to be relevant by $\pi_0$ are shown and observed; items estimated irrelevant are neither shown nor observed.

**Effect on retrieval evaluation:** In two-stage retrieval+ranking systems, only items that the retrieval stage surfaces can be evaluated. If the retrieval stage uses a fixed embedding model or BM25, items outside its semantic neighborhood are **permanently invisible** to the evaluation pipeline. Evaluating a new retrieval model against logged data will systematically underestimate its quality for novel/OOD items it retrieves that the logging retrieval never would have found.

**The missing data structure:**

| | $M_{u,i} = 1$ (shown) | $M_{u,i} = 0$ (not shown) |
|---|---|---|
| Relevant | Observed ✓ | **Missing — assumed irrelevant** |
| Irrelevant | Observed ✓ | Missing — assumed irrelevant |

The problem is that the bottom-left cell (relevant but not shown) is treated identically to the bottom-right cell (irrelevant and not shown). This is the MNAR assumption violation.

## 1.3 The Fundamental Problem of Counterfactual Evaluation

The counterfactual evaluation problem is formally equivalent to the **fundamental problem of causal inference** (Holland, 1986): you cannot observe both the potential outcome under $\pi_0$ and the potential outcome under $\pi_e$ for the same unit.

Let $Y_u(\mathbf{a})$ denote the potential outcome (e.g., total clicks) if user $u$ were shown slate $\mathbf{a}$. The policy value is:

$$V(\pi) = \mathbb{E}_{u}\left[\mathbb{E}_{\mathbf{a} \sim \pi}[Y_u(\mathbf{a})]\right]$$

From the log, we observe $Y_u(\mathbf{a}_0)$ where $\mathbf{a}_0 \sim \pi_0$. We want $V(\pi_e)$ but can only observe outcomes under $\pi_0$.

The direct estimator naively estimates:

$$\hat{V}_{\text{DM}}(\pi_e) = \frac{1}{|\mathcal{D}|} \sum_{(u, \mathbf{a}_0, y) \in \mathcal{D}} \hat{r}(u, \pi_e(u))$$

where $\hat{r}$ is a learned reward model. This is only unbiased if the reward model is perfectly specified — which it never is.

The **policy gap** between $\pi_0$ and $\pi_e$ is the fraction of $\pi_e$'s output that $\pi_0$ would never produce:

$$\delta(\pi_0, \pi_e) = \mathbb{E}_u[P_{\mathbf{a} \sim \pi_e}(\mathbf{a} \notin \text{support}(\pi_0(\cdot|u)))]$$

When $\delta > 0$ (which is always true for a genuinely new model), there are slates that $\pi_e$ would produce for which we have **zero logged data**. No debiasing technique resolves this — it is a fundamental extrapolation problem.

**Practical implication:** The more different your new model is from the logging policy, the less reliable your offline evaluation becomes, independent of which debiasing technique you use. This is why truly novel architectures (e.g., switching from a matrix factorization retrieval model to a dense neural retrieval model) always need live A/B testing — offline evaluation cannot validate them reliably.

## 1.4 Implicit vs. Explicit Feedback: The Mathematical Distinction

### Explicit Feedback

Explicit feedback is a direct relevance signal provided by the user: star ratings, thumbs up/down, or numerical scores. Mathematically, if we model user $u$'s true preference for item $i$ as $r_{u,i} \in \mathbb{R}$, then explicit feedback $y_{u,i}^{\text{exp}}$ satisfies:

$$y_{u,i}^{\text{exp}} = r_{u,i} + \epsilon_{u,i}, \quad \epsilon_{u,i} \sim \mathcal{N}(0, \sigma^2)$$

The key property: **$y_{u,i}^{\text{exp}}$ is a noisy but unbiased estimator of $r_{u,i}$**. With enough data, you can recover the true preference structure. The MNAR problem still exists (users only rate items they chose to interact with), but the signal itself is interpretable.

### Implicit Feedback

Implicit feedback (clicks, dwell time, purchases, streams) is an **indirect and confounded proxy** for preference. The data-generating process is:

$$c_{u,i} = o_{u,i} \cdot \mathbb{1}[r_{u,i} > \tau_{u,i}]$$

where:
- $c_{u,i} \in \{0, 1\}$: observed click
- $o_{u,i} \in \{0, 1\}$: whether the item was examined (latent variable, depends on position and attention)
- $r_{u,i}$: true relevance (latent)
- $\tau_{u,i}$: click threshold (also latent — depends on user state, query, context)

**Why this changes everything:**

1. **A click is not a preference signal — it is an examination × preference signal.** $\mathbb{E}[c_{u,i} \mid o_{u,i}=1] = P(r_{u,i} > \tau_{u,i})$ is a noisy relevance estimate. But $\mathbb{E}[c_{u,i}] = P(o_{u,i}=1) \cdot P(r_{u,i} > \tau_{u,i})$, which is confounded by examination probability.

2. **A non-click is not a negative signal.** Under implicit feedback, $c_{u,i} = 0$ means either the item was not examined OR the item was examined and found irrelevant. These are very different situations that require completely different treatments in a loss function.

3. **The threshold $\tau_{u,i}$ is context-dependent.** Users have different click propensities depending on their query intent, current mood, device type, and the quality of the surrounding results (the cascade effect: if the first result is excellent, the user may not click subsequent ones even if they are also good).

**Consequence for loss function design:**

The standard binary cross-entropy loss:

$$\mathcal{L} = -\sum_{u,i} [c_{u,i} \log \hat{r}_{u,i} + (1 - c_{u,i}) \log(1 - \hat{r}_{u,i})]$$

treats non-clicks as negative examples. This is correct only for items that were examined but not clicked — for unexamined items, including them as negatives introduces systematic downward bias on item quality estimates. This is why techniques like **sampled softmax** (which treats non-shown items as negatives via sampling) must be used carefully, and why **exposure-weighted** loss functions are theoretically preferred.

---

## Level 1 Study Aids

### Concept Check

**Q1.** You are evaluating a new retrieval model offline. The logging policy is BM25. Your new model uses dense vector retrieval and retrieves many items that BM25 would never retrieve. You compute Recall@100 on a held-out split of logged data. Your new model scores *lower* than BM25. Is this evidence that the new model is worse? Construct a rigorous argument for why this comparison might be systematically flawed, and propose what additional data collection strategy would fix it.

**Q2.** Suppose you observe that two models, A and B, achieve identical NDCG@10 on your offline evaluation set. Model A is a small modification of the logging policy; model B is a completely different architecture that ranks items very differently. Which model's offline NDCG estimate do you trust more, and why? Formalize your answer in terms of policy gap $\delta(\pi_0, \pi)$.

**Q3.** A colleague proposes to remove position bias by stratifying evaluation: compute NDCG separately for items shown at each rank position, then average. Does this remove position bias? What does it actually measure, and when is it valid?

---

### Common Misconceptions

**Misconception 1: "We avoid position bias by shuffling our training data."**

Shuffling the training data changes the order in which gradient updates are applied, not the content of the labels. Position bias is baked into the labels themselves: a click on position 1 was generated by a user with examination probability $\theta_1$, and a click on position 5 was generated with $\theta_5 < \theta_1$. Shuffling training examples does nothing to correct this label-level confounding.

**Misconception 2: "High Recall@K means our system is finding the relevant items."**

High Recall@K on a biased log means your system is finding the items that were *previously shown and clicked*, which skews toward popular items. A system that recommends the top-200 most popular items globally will achieve high Recall@K on most biased datasets. True recall requires knowing the full set of relevant items for each user, which logged data cannot provide.

**Misconception 3: "Implicit feedback is fine as long as you have enough of it."**

More biased data does not reduce bias — it entrenches it. The position bias, popularity bias, and selection bias in your feedback are structural: they arise from the system architecture, not from data insufficiency. A billion biased clicks is still a billion biased clicks. Quantity resolves variance; it cannot resolve systematic bias.

---

### Connections to Adjacent Levels

- **→ Level 2:** The biases described here corrupt every metric in Level 2. Position bias makes raw NDCG computed from click data unreliable; popularity bias inflates Recall@K; selection bias means MAP computed from logged data is a biased estimate of true MAP. Before using any metric, you must understand which bias regime you are operating in.

- **→ Level 3:** The debiasing techniques in Level 3 are direct responses to the biases catalogued here. IPS corrects for selection bias; position-based models correct for position bias; popularity debiasing addresses the long-tail problem. Each technique makes an assumption that neutralizes one of the bias mechanisms described in this level — but not all of them simultaneously.

- **→ Level 4:** The offline-online gap (Level 4) is largely a consequence of the counterfactual evaluation problem (Section 1.3). When your offline eval is computed from data generated by $\pi_0$ and you want to estimate the value of $\pi_e$, the reliability of the offline estimate degrades as a function of $\delta(\pi_0, \pi_e)$.

---

<a id='level-2'></a>
# LEVEL 2: CORE METRICS — DERIVATIONS AND TRADEOFFS

---

## 2.1 Precision@K and Recall@K

### Derivation

Let $\mathcal{R}(q)$ be the set of relevant items for query $q$, and $S_K(q)$ be the top-K retrieved items. Then:

$$\text{Precision@K}(q) = \frac{|S_K(q) \cap \mathcal{R}(q)|}{K}$$

$$\text{Recall@K}(q) = \frac{|S_K(q) \cap \mathcal{R}(q)|}{|\mathcal{R}(q)|}$$

Averaged over a query set $Q$:

$$\text{P@K} = \frac{1}{|Q|} \sum_{q \in Q} \text{Precision@K}(q), \quad \text{R@K} = \frac{1}{|Q|} \sum_{q \in Q} \text{Recall@K}(q)$$

### Intuition

Precision@K asks: *Of the K items I showed the user, how many were actually relevant?* It is a user-experience metric — a measure of how much time the user wastes on irrelevant results.

Recall@K asks: *Of all the relevant items that exist, how many did I surface within the top K?* It is a coverage metric — a measure of how much of the relevant space the system captures.

These two metrics are in fundamental tension. Increasing K always weakly increases Recall@K and weakly decreases Precision@K. The tradeoff is governed by the precision-recall curve.

### Failure Modes

1. **Rank-blind:** Both metrics treat all K positions equally. An item at rank 1 and rank K contribute identically to P@K. A system that finds all relevant items but puts them at the bottom of the list gets perfect Recall@K and perfect Precision@K — but is objectively worse than a system that puts relevant items first. This is why rank-weighted metrics (DCG, MRR) were developed.

2. **Precision@K depends on K:** A system optimized for P@5 is not necessarily the best system for P@20. You must choose K based on your UI constraint (how many results does your interface show?).

3. **Recall@K requires knowing $|\mathcal{R}(q)|$:** In production log data, you only know about items the user clicked — a severe undercount of truly relevant items. This makes Recall@K computed from logs a badly biased metric.

### When to Use

Use Precision@K when: your UI displays exactly K items and user satisfaction scales with the fraction that are relevant (e.g., a recommendation carousel with 5 slots). Use Recall@K when: you care about not missing relevant items and the cost of a miss is high (e.g., legal document retrieval, medical literature search).

## 2.2 Average Precision and Mean Average Precision

### Derivation

Average Precision (AP) for query $q$ integrates precision over all recall levels where a relevant document is found. Let $\text{rel}(k) \in \{0, 1\}$ be the relevance of the item at rank $k$. Then:

$$\text{AP}(q) = \frac{1}{|\mathcal{R}(q)|} \sum_{k=1}^{K} \text{Precision@k}(q) \cdot \text{rel}(k)$$

Expanded:

$$\text{AP}(q) = \frac{1}{|\mathcal{R}(q)|} \sum_{k=1}^{K} \frac{\sum_{j=1}^{k} \text{rel}(j)}{k} \cdot \text{rel}(k)$$

This computes Precision@k **at each rank where a relevant document occurs**, then averages over relevant documents. Items at positions where $\text{rel}(k) = 0$ do not contribute.

Mean Average Precision:

$$\text{MAP} = \frac{1}{|Q|} \sum_{q \in Q} \text{AP}(q)$$

### Intuition

AP is the area under the precision-recall curve (with some approximation). It rewards systems that put *all* relevant documents near the top, not just the first one. A system that puts 10 relevant documents at ranks 1–10 gets higher AP than one that puts them at ranks 1, 11, 21, ..., 91 — even though both have Recall@100 = 1.0.

The $1/|\mathcal{R}(q)|$ normalization ensures AP ∈ [0, 1] and is comparable across queries with different numbers of relevant documents.

### Failure Modes

1. **Binary relevance only:** AP is derived assuming binary $\text{rel}(k) \in \{0,1\}$. For graded relevance (e.g., highly relevant, somewhat relevant, irrelevant), use NDCG.

2. **Sensitive to the number of relevant documents:** If $|\mathcal{R}(q)|$ is small, AP has high variance. A query with 1 relevant document has AP = 1/rank_of_first_relevant — a tiny difference in the ranking of that one document swings AP dramatically.

3. **Requires complete relevance judgments:** MAP is undefined (or severely biased) when $\mathcal{R}(q)$ is estimated from incomplete logs.

### When to Use

MAP is the standard metric for information retrieval benchmarks (TREC, MS MARCO) where complete relevance assessments exist. It is the metric of choice when: (a) you have binary relevance, (b) you have complete or nearly complete relevance judgments, and (c) you care about ranking all relevant documents well, not just the first.

## 2.3 Reciprocal Rank and Mean Reciprocal Rank

### Derivation

For query $q$, let $\text{rank}_q$ be the rank of the first relevant item:

$$\text{RR}(q) = \frac{1}{\text{rank}_q}$$

If no relevant item appears in the top K, $\text{RR}(q) = 0$.

$$\text{MRR} = \frac{1}{|Q|} \sum_{q \in Q} \frac{1}{\text{rank}_q}$$

### Intuition

MRR captures only one thing: how quickly the user finds *the first* relevant item. It implicitly assumes a cascade user model where the user scans from the top and stops as soon as they find something useful. For queries with a single correct answer (navigational queries, factual lookups), this is exactly the right metric.

### When MRR Misleads

MRR is unreliable when:

1. **The query has multiple relevant items.** MRR completely ignores items beyond the first relevant one. Two systems with identical MRR may differ dramatically in how many relevant items they surface overall.

2. **The query distribution is mixed.** If your query set contains both navigational queries (one correct answer) and exploratory queries (many relevant documents), MRR conflates two fundamentally different evaluation regimes. The navigational queries will dominate MRR because they have cleaner signals.

3. **The rank distribution has heavy tails.** MRR is extremely sensitive to whether the first relevant item is at rank 1 vs. rank 2 (RR goes from 1.0 to 0.5), but relatively insensitive to rank 5 vs. rank 10 (RR goes from 0.2 to 0.1). This makes MRR volatile for queries where the first relevant item tends to appear deep in the ranking.

## 2.4 DCG and NDCG@K: Full Derivation

### The Motivation for Logarithmic Discounting

The key insight is that a user's probability of examining a document at rank $k$ decreases with $k$, and this decrease is approximately logarithmic with rank. The **Position-Based Model** (Craswell et al., 2008) posits:

$$P(\text{click} \mid k, r) = P(\text{examine} \mid k) \cdot P(\text{click} \mid \text{examine}, r) \approx \frac{1}{\log_2(k+1)} \cdot r$$

A metric that measures expected utility under this examination model should weight relevance at rank $k$ by $1/\log_2(k+1)$.

### DCG Derivation

**Discounted Cumulative Gain:**

$$\text{DCG@K} = \sum_{k=1}^{K} \frac{\text{rel}_k}{\log_2(k+1)}$$

where $\text{rel}_k \in \{0, 1, 2, 3, 4\}$ is the graded relevance of the item at rank $k$.

The alternative formulation (used by many industry systems and preferred when relevance grades are sparse):

$$\text{DCG@K} = \sum_{k=1}^{K} \frac{2^{\text{rel}_k} - 1}{\log_2(k+1)}$$

This version exponentiates the relevance, which **emphasizes the difference between high grades** (e.g., rel=3 vs. rel=4 gets amplified more than rel=1 vs. rel=2). For binary relevance, $2^{\text{rel}} - 1 = \text{rel}$ and both formulations are identical.

**Why log base 2?** It is purely conventional. Log base 2 means that an item at rank 2 is discounted by exactly $1/\log_2(3) \approx 0.63$ relative to rank 1. The base determines how steep the discount curve is but does not change the ranking of systems. Base $e$ and base 10 are sometimes used; the choice only affects the absolute scale of DCG, not relative comparisons between systems.

### Relevance Grades from Binary Click Signal

When you only have binary click data ($c_{u,i} \in \{0, 1\}$), you typically assign:
- $\text{rel} = 1$ if clicked
- $\text{rel} = 0$ if not clicked

This reduces DCG to a position-weighted click count. More sophisticated approaches use:
- **Dwell time tiers:** $\text{rel} = 0, 1, 2$ based on whether the user spent 0, 10–60, or >60 seconds
- **Engagement depth:** $\text{rel} = 0, 1, 2, 3$ based on no click, click without conversion, add-to-cart, purchase
- **Explicit labels:** Human rater judgments on a 5-point scale (used in TREC/academic benchmarks)

The mapping from observed signals to relevance grades is a **design decision** with significant consequences. A wrong mapping (e.g., treating a 1-second accidental click identically to a 10-minute session) will distort the metric.

### Normalization: NDCG

DCG is unnormalized: it depends on the number of relevant documents and the grading scale. NDCG normalizes by the **Ideal DCG** (IDCG) — the DCG of the perfect ranking:

$$\text{IDCG@K}(q) = \sum_{k=1}^{\min(K, |\mathcal{R}(q)|)} \frac{\text{rel}_{(k)}}{\log_2(k+1)}$$

where $\text{rel}_{(k)}$ denotes the $k$-th largest relevance score among all relevant items.

$$\text{NDCG@K}(q) = \frac{\text{DCG@K}(q)}{\text{IDCG@K}(q)}$$

NDCG@K $\in [0, 1]$ for any query with at least one relevant document, enabling averaging across queries. **Important edge case:** if a query has no relevant documents, IDCG = 0 and NDCG is undefined. Such queries must be excluded from the evaluation set.

### Failure Modes

1. **NDCG assumes a static user model.** The logarithmic discount assumes position-based examination. Users who skip to interesting titles, use Ctrl+F, or scroll non-linearly are not captured by this model.

2. **NDCG is insensitive to recall.** A system that retrieves only 1 perfectly relevant item at rank 1 can achieve NDCG@10 = 1.0, even though it missed 9 other relevant items (assuming the normalization only counts up to the number of items retrieved).

3. **NDCG from click data inherits all biases from Level 1.** Click-based NDCG is position-biased, popularity-biased, and selection-biased.

## 2.5 Expected Reciprocal Rank (ERR)

### Motivation

NDCG and MRR both assume that users examine positions independently. ERR models a more realistic **cascade** user who examines results sequentially and stops when satisfied. The probability of examining position $k$ depends on the user *not* being satisfied by positions 1 through $k-1$.

### Derivation

Let $R_k \in [0,1]$ be the **graded relevance probability** at rank $k$ — the probability that the user is satisfied by the item at rank $k$. For graded labels $g_k \in \{0, 1, 2, 3, 4\}$:

$$R_k = \frac{2^{g_k} - 1}{2^{g_{\max}}}$$

The probability that the user examines position $k$ is:

$$P(\text{examine position } k) = \prod_{j=1}^{k-1} (1 - R_j)$$

This is the probability of **not stopping** at any earlier position. The probability that the user stops exactly at position $k$ is:

$$P(\text{stop at } k) = R_k \cdot \prod_{j=1}^{k-1}(1 - R_j)$$

ERR is the **expected reciprocal rank at which the user stops**:

$$\text{ERR} = \sum_{k=1}^{K} \frac{1}{k} \cdot R_k \cdot \prod_{j=1}^{k-1}(1 - R_j)$$

### How ERR Differs from MRR

MRR treats $R_k \in \{0, 1\}$ (binary relevance) and finds the first relevant item, ignoring continuation probability. ERR allows graded satisfaction and explicitly models the probability of continuing past a partially satisfying result.

Example: A result with $R_1 = 0.5$ ("somewhat relevant") reduces but does not eliminate examination of position 2. In MRR, if $g_1 \geq 1$ you stop — ERR handles the more realistic case where users sometimes continue browsing even after a good result.

### When ERR is the More Correct Metric

ERR is superior to MRR when:
1. You have graded relevance (not just binary)
2. The user task has a strong cascade structure (navigational search, Q&A)
3. You expect that very good results early in the list change the user's behavior toward subsequent results

For multi-document tasks where users consume multiple results (recommendation feeds, playlist generation), neither ERR nor MRR is appropriate — DCG/NDCG with a cumulative value model is better.

## 2.6 Hit Rate@K vs. Recall@K

### Definitions

$$\text{HitRate@K} = \frac{1}{|Q|} \sum_{q \in Q} \mathbb{1}[\mathcal{R}(q) \cap S_K(q) \neq \emptyset]$$

$$\text{Recall@K} = \frac{1}{|Q|} \sum_{q \in Q} \frac{|\mathcal{R}(q) \cap S_K(q)|}{|\mathcal{R}(q)|}$$

### When Are They Equivalent?

**They are equivalent when $|\mathcal{R}(q)| = 1$ for all queries.** If each query has exactly one relevant item (leave-one-out evaluation, where you hold out one interaction per user and treat it as the ground truth), then:

$$\text{Recall@K}(q) = \frac{|\{i^*\} \cap S_K(q)|}{1} = \mathbb{1}[i^* \in S_K(q)] = \text{HitRate@K}(q)$$

This is the most common evaluation setup in academic recommendation research (e.g., on MovieLens, Amazon Reviews): hold out the last interaction, evaluate whether it appears in the top K recommendations.

### When Are They Not Equivalent?

When $|\mathcal{R}(q)| > 1$:
- Hit Rate@K asks: "Did we find *at least one* relevant item?"
- Recall@K asks: "What fraction of *all* relevant items did we find?"

A system that finds 1 out of 10 relevant items achieves HitRate@K = 1.0 (if that item is in S_K) but Recall@K = 0.1. These measure fundamentally different things.

**Practical guidance:** Use HitRate@K for evaluation setups that simulate a single-session interaction ("did the user find anything useful?"). Use Recall@K when you care about systematic coverage of the relevant space.

## 2.7 Beyond-Accuracy Metrics

### Coverage

$$\text{Coverage} = \frac{|\bigcup_{u \in \mathcal{U}} S_K(u)|}{|\mathcal{I}|}$$

Coverage measures the fraction of the item catalog that the system recommends to at least one user. A system with Coverage = 0.02 is surfacing only 2% of the catalog — the rest is permanently invisible to users, regardless of quality.

**Business relevance:** In marketplace settings (Etsy, Airbnb), low coverage means sellers/hosts with niche offerings never get traffic, even when their items are highly relevant to specific users. This creates a systematic bias against long-tail supply.

Coverage does not measure quality — a random system has high coverage. It must be tracked alongside accuracy metrics.

### Intra-List Diversity (ILD)

$$\text{ILD}(S_K(u)) = \frac{1}{K(K-1)} \sum_{i \in S_K} \sum_{j \in S_K, j \neq i} d(i, j)$$

where $d(i, j)$ is a dissimilarity measure between items $i$ and $j$ in some feature space (genre vectors, embedding cosine distance, etc.).

ILD measures how different the items in a recommendation list are from each other. High ILD means the list covers a broad range of the item space; low ILD means the list is a cluster of similar items.

**Failure mode:** ILD is maximized by recommending maximally different items — but maximally different does not mean maximally useful. A movie recommendation list containing horror, children's animation, documentary, and adult comedy has high ILD but is probably wrong for any specific user. ILD must be conditioned on *relevant* items only to be meaningful.

### Novelty

$$\text{Novelty}(S_K(u)) = -\frac{1}{K} \sum_{i \in S_K(u)} \log_2 P(i)$$

where $P(i)$ is the global popularity of item $i$ (fraction of users who interacted with it). Novelty is the self-information (surprise) of the recommendation. High novelty means the system is recommending rare items; low novelty means it is recommending only blockbusters.

**Connection to business value:** Long-tail novelty is commercially valuable because long-tail items have lower competition (other systems also recommend popular items) and can drive discovery. However, novelty without relevance is useless — a system recommending random obscure items has maximum novelty and zero value.

### Serendipity

Serendipity is the hardest to define formally. One formulation:

$$\text{Serendipity}(S_K(u)) = \frac{1}{K} \sum_{i \in S_K(u)} \text{Unexpected}(i, u) \cdot \text{Relevant}(i, u)$$

where:
- $\text{Unexpected}(i, u) = 1 - \text{CosSim}(\mathbf{e}_i, \bar{\mathbf{e}}_u)$: how far item $i$ is from the user's historical preference centroid
- $\text{Relevant}(i, u)$: whether the item is actually relevant (measured by future interaction)

Serendipity requires items to be **both unexpected and relevant** — discovery of something outside the user's typical patterns that they genuinely enjoy. This is very hard to measure offline because the "relevant" component requires future signal.

**Business relevance:** Serendipity correlates with long-term user retention. Users who only see predictable recommendations experience the "filter bubble" effect and churn. Systems optimized purely for short-term CTR sacrifice serendipity and hurt long-term engagement (this is well-documented in the Spotify and Netflix literature).

## 2.8 Disagreements, Failure Cases, and Goodhart's Law

### When NDCG Disagrees with MAP

NDCG and MAP disagree when **the distribution of relevant items across ranks differs** even though aggregate quality is similar.

**Concrete example:** Query has 4 relevant items out of 10 retrieved.

System A ranking: [R, R, I, I, I, R, I, I, I, R] (R=relevant, I=irrelevant)

System B ranking: [R, I, R, I, I, I, R, I, I, R]

Computing MAP:
- System A: AP = (1/4)[P@1·1 + P@2·1 + P@6·1 + P@10·1] = (1/4)[1 + 1 + 0.5 + 0.4] = 0.725
- System B: AP = (1/4)[P@1·1 + P@3·1 + P@7·1 + P@10·1] = (1/4)[1 + 0.667 + 0.429 + 0.4] = 0.624

Computing NDCG@10 (using $\text{rel}/\log_2(k+1)$, IDCG = sum of top-4 discounts):
- IDCG = 1/1 + 1/1.585 + 1/2 + 1/2.322 = 1 + 0.631 + 0.5 + 0.431 = 2.562
- System A DCG = 1/1 + 1/1.585 + 1/3.459 + 1/4.322 = 1 + 0.631 + 0.289 + 0.231 = 2.151, NDCG = 0.839
- System B DCG = 1/1 + 1/2.322 + 1/3.459 + 1/4.322 = 1 + 0.431 + 0.289 + 0.231 = 1.951, NDCG = 0.761

Both metrics agree that A > B, but the magnitudes differ because MAP's precision computation weights consecutive relevant results differently than NDCG's log discount.

**Where they truly disagree:** When relevant items are clustered vs. spread. MAP rewards spreading relevant items across the ranking (each new relevant item at position $k$ contributes $1/k$ to AP via precision); NDCG rewards clustering relevant items at the top due to the steeper discount curve. A system that frontloads all relevant items may be penalized by MAP (P@k drops after the cluster) but rewarded by NDCG.

**My opinion:** For production recommendation systems, NDCG is the more useful metric because it directly models the value of top-of-list relevance. MAP is better for IR benchmarks with complete relevance assessments where you care equally about all relevant documents.

### Goodhart's Law in Recommenders

Goodhart's Law: *When a measure becomes a target, it ceases to be a good measure.*

In recommender systems, this manifests as **recommendation collapse**: optimizing a single metric causes the system to find shortcuts that improve the metric without improving the underlying user experience.

**CTR collapse example:** A system optimized for click-through rate will learn to recommend clickbait — items with misleading thumbnails or titles that generate clicks but poor downstream engagement. CTR improves; user satisfaction degrades.

**NDCG collapse example:** Optimizing NDCG against a popularity-biased training set causes the system to concentrate recommendations on the head of the popularity distribution (where training signal is densest), collapsing coverage and novelty.

**Diversity collapse example:** A system that learns a very accurate user preference model may produce recommendation lists with near-zero diversity — every item is of the same type because the model correctly identifies the user's dominant preference, missing opportunities for serendipitous discovery.

**The structural fix:** Multi-objective evaluation using a **dashboard of metrics** (accuracy + diversity + novelty + coverage + business KPIs) rather than a single scalar. The combination of metrics is harder to Goodhart because improving one at the expense of others becomes visible.

---

## Level 2 Study Aids

### Concept Check

**Q1.** You have two retrieval systems. System A achieves NDCG@10 = 0.72 and Coverage = 0.05. System B achieves NDCG@10 = 0.68 and Coverage = 0.45. In a marketplace recommendation context where seller fairness matters, which system do you deploy, and what additional metrics would you want to measure before deciding?

**Q2.** Your team is evaluating a new ranking model against a leave-one-out test set on MovieLens-1M. You observe HitRate@10 = 0.78. Your manager asks: "Does this mean 78% of users found what they were looking for?" Construct a precise explanation of what HitRate@10 = 0.78 means and what it does not mean, given the evaluation setup.

**Q3.** Derive formally why the ERR of a ranking in which all items have maximum relevance $R_k = 1$ equals 1.0 (you should be able to compute this from the ERR formula directly). Then explain why a ranking with $R_1 = 0.5$ and $R_2 = 1.0$ has lower ERR than $R_1 = 1.0$ and $R_2 = 0.5$.

---

### Common Misconceptions

**Misconception 1: "NDCG@K = 1 means our ranker is perfect."**

NDCG@K = 1 means the system produced the *ideal ranking of the items it retrieved* with respect to the measured relevance signals. It says nothing about whether the right items were retrieved in the first place (retrieval quality), whether the relevance labels are accurate (label quality), or whether the metric captures what users actually care about (metric validity).

**Misconception 2: "Diversity metrics are just nice-to-have."
**
Diversity metrics are **business-critical** in several settings: (a) marketplace platforms where they operationalize supply fairness; (b) news and media where filter bubbles have reputational and social costs; (c) long-session applications (music, video) where monotone recommendations cause session abandonment. Treating diversity as a tie-breaker rather than a primary constraint is a product mistake.

**Misconception 3: "Higher K is always better for evaluation."**

NDCG@100 is not a better metric than NDCG@10 if your UI only shows 10 results. Evaluation at K should match the K at which users actually experience results. Evaluating at large K inflates the apparent quality of systems that put relevant items deep in the ranking, which never benefits real users.

---

### Connections

- **← Level 1:** All metrics computed from biased logs inherit the biases described in Level 1. The metrics are mathematically clean; their inputs are not. Position bias corrupts DCG computed from click data; popularity bias corrupts Recall@K; selection bias means IDCG is computed over a filtered item set.

- **→ Level 3:** The debiasing techniques in Level 3 aim to produce unbiased estimates of the metrics defined here. IPS-weighted NDCG, for example, is an attempt to estimate "what NDCG would be if items were shown uniformly" rather than according to $\pi_0$.

- **→ Level 4:** The offline-online gap is partly a consequence of optimizing for metrics that do not predict online behavior. The gap between NDCG and CTR lift is a measurement validity problem: NDCG under a specific examination model may not capture the user experience factors (freshness, diversity, trust) that drive live engagement.

---

<a id='level-3'></a>
# LEVEL 3: DEBIASING TECHNIQUES

---

## 3.1 Inverse Propensity Scoring (IPS)

### Derivation from Importance Weighting

The core problem: we want to estimate the expected reward under evaluation policy $\pi_e$ but can only observe rewards under logging policy $\pi_0$. This is the **off-policy evaluation** problem.

The population-level quantity we want:

$$V(\pi_e) = \mathbb{E}_{u \sim \mathcal{U}}\mathbb{E}_{\mathbf{a} \sim \pi_e(\cdot|u)}[R(u, \mathbf{a})]$$

The data we have: $\mathcal{D} = \{(u_t, \mathbf{a}_t, r_t)\}_{t=1}^{n}$ where $\mathbf{a}_t \sim \pi_0(\cdot|u_t)$.

By importance weighting (the Radon-Nikodym trick), if $\pi_0$ has full support over $\pi_e$ (every action $\pi_e$ would take, $\pi_0$ takes with positive probability):

$$V(\pi_e) = \mathbb{E}_{u \sim \mathcal{U}}\mathbb{E}_{\mathbf{a} \sim \pi_e}[R(u, \mathbf{a})] = \mathbb{E}_{u}\mathbb{E}_{\mathbf{a} \sim \pi_0}\left[\frac{\pi_e(\mathbf{a}|u)}{\pi_0(\mathbf{a}|u)} R(u, \mathbf{a})\right]$$

The ratio $w(u, \mathbf{a}) = \pi_e(\mathbf{a}|u)/\pi_0(\mathbf{a}|u)$ is the **importance weight**. Its reciprocal $1/\pi_0(\mathbf{a}|u)$ is the **inverse propensity score**.

The IPS estimator:

$$\hat{V}_{\text{IPS}}(\pi_e) = \frac{1}{n} \sum_{t=1}^{n} \frac{\pi_e(\mathbf{a}_t | u_t)}{\pi_0(\mathbf{a}_t | u_t)} r_t$$

**Unbiasedness proof:**

$$\mathbb{E}[\hat{V}_{\text{IPS}}] = \mathbb{E}_{u}\mathbb{E}_{\mathbf{a} \sim \pi_0}\left[\frac{\pi_e(\mathbf{a}|u)}{\pi_0(\mathbf{a}|u)} r(u, \mathbf{a})\right] = \mathbb{E}_{u}\sum_{\mathbf{a}} \pi_e(\mathbf{a}|u) r(u, \mathbf{a}) = V(\pi_e) \checkmark$$

### What is the Propensity Score in Recommendations?

In search/recommendation, we rarely have clean stochastic policies. The **propensity score** $\pi_0(i | u, q)$ is the probability that item $i$ was shown to user $u$ in response to query $q$ under the logging policy. This is:

1. **In a stochastic logging policy:** directly computable from the policy's probability distribution
2. **In a deterministic logging policy:** requires estimating propensities from system logs (e.g., from position CTR experiments)
3. **For individual items in a slate:** often approximated as $\pi_0(i | u, q) \approx P(\text{position} \leq K | u, q) \cdot P(i | \text{retrieved} )$

In practice, propensity estimation is the hardest part of IPS. If your propensities are wrong, your IPS estimate is biased.

### The High Variance Problem

The variance of the IPS estimator:

$$\text{Var}[\hat{V}_{\text{IPS}}] = \frac{1}{n} \text{Var}_{\mathbf{a} \sim \pi_0}\left[\frac{\pi_e(\mathbf{a}|u)}{\pi_0(\mathbf{a}|u)} r(u, \mathbf{a})\right]$$

When $\pi_e$ and $\pi_0$ are very different (large policy gap), the importance weights $\pi_e/\pi_0$ can be enormous for actions that $\pi_e$ assigns high probability but $\pi_0$ assigns low probability. In the extreme case where $\pi_0(\mathbf{a}|u) \approx 0$ but $\pi_e(\mathbf{a}|u) > 0$, the weight blows up to infinity.

More formally, the variance is bounded by:

$$\text{Var}[\hat{V}_{\text{IPS}}] \leq \frac{1}{n} \left(\max_{(u,\mathbf{a})} \frac{\pi_e(\mathbf{a}|u)}{\pi_0(\mathbf{a}|u)}\right)^2 \cdot \mathbb{E}[r^2]$$

The maximum importance weight can be exponential in the action space dimensionality — for slates of size $K$ drawn from $N$ items, the support mismatch compounds multiplicatively.

### Clipping and the Bias-Variance Tradeoff

**Clipped IPS:** Truncate importance weights at a maximum value $M$:

$$\hat{V}_{\text{IPS-clip}}(\pi_e) = \frac{1}{n} \sum_{t=1}^{n} \min\left(M, \frac{\pi_e(\mathbf{a}_t | u_t)}{\pi_0(\mathbf{a}_t | u_t)}\right) r_t$$

**Effect:** Clipping introduces bias (we underweight high-importance samples) but reduces variance. The optimal $M$ minimizes mean squared error:

$$\text{MSE} = \text{Bias}^2 + \text{Variance}$$

As $M \to \infty$, bias → 0 and variance → ∞. As $M \to 0$, bias → large and variance → 0. The optimal $M^*$ depends on the specific distribution of importance weights — in practice, $M$ is tuned via cross-validation on a held-out slice of logs.

**My opinion:** In production systems, clipping is almost always necessary. Unclipped IPS is theoretically correct but practically unreliable — a single high-weight event can dominate the entire estimate. I recommend $M = \text{95th percentile of observed importance weights}$ as a reasonable starting point, with sensitivity analysis around this value.

## 3.2 Self-Normalized IPS (SNIPS)

### Derivation

The SNIPS estimator (Swaminathan & Joachims, 2015b) normalizes the importance weights by their sum:

$$\hat{V}_{\text{SNIPS}}(\pi_e) = \frac{\sum_{t=1}^{n} \frac{\pi_e(\mathbf{a}_t | u_t)}{\pi_0(\mathbf{a}_t | u_t)} r_t}{\sum_{t=1}^{n} \frac{\pi_e(\mathbf{a}_t | u_t)}{\pi_0(\mathbf{a}_t | u_t)}}$$

This can be written as a weighted average:

$$\hat{V}_{\text{SNIPS}} = \sum_{t=1}^{n} \tilde{w}_t \cdot r_t, \quad \tilde{w}_t = \frac{w_t}{\sum_{s=1}^{n} w_s}, \quad w_t = \frac{\pi_e(\mathbf{a}_t|u_t)}{\pi_0(\mathbf{a}_t|u_t)}$$

### Why Lower Variance

SNIPS is a **ratio estimator**. The denominator $\hat{Z} = \frac{1}{n}\sum_t w_t$ is itself an estimate of $\mathbb{E}_{\pi_0}[w_t] = 1$ (since importance weights average to 1 under $\pi_0$). By normalizing, we effectively correct for cases where the realized weights are unusually high or low due to finite-sample variability.

The variance reduction comes from the **negative covariance** between numerator and denominator: when the numerator is high due to large weights, the denominator is also high, and the ratio is more stable than either individually.

Formally, using the delta method for ratio estimators, SNIPS has variance:

$$\text{Var}[\hat{V}_{\text{SNIPS}}] \approx \frac{1}{n} \text{Var}[w_t r_t - V(\pi_e) w_t]$$

which is typically smaller than $\text{Var}[\hat{V}_{\text{IPS}}] = \frac{1}{n}\text{Var}[w_t r_t]$ because the $-V(\pi_e) w_t$ term reduces variance when $r_t$ and $w_t$ are positively correlated.

### When SNIPS is Biased and IPS is Not

SNIPS is **biased** for any finite sample size. The bias comes from the ratio of two expectations:

$$\mathbb{E}\left[\frac{A}{B}\right] \neq \frac{\mathbb{E}[A]}{\mathbb{E}[B]}$$

The SNIPS bias is $O(1/n)$ — it vanishes asymptotically but can be significant with small samples.

**When does SNIPS bias matter?**
1. Small datasets ($n < 1000$ events per query/user stratum)
2. Very high variance importance weights where the ratio approximation breaks down
3. When you need an **unbiased** certificate for regulatory or audit purposes

**My opinion:** In practice, SNIPS is almost always preferable to IPS. The $O(1/n)$ bias is negligible at production data scales, and the variance reduction is substantial. I have never seen a production system where the SNIPS bias was the dominant source of estimation error — propensity estimation error always dominates.

## 3.3 Doubly Robust (DR) Estimators

### Derivation

The DR estimator (Dudík et al., 2011; Saito & Joachims, 2021 for ranking) combines a **Direct Method (DM)** reward model with an **IPS correction term**:

$$\hat{V}_{\text{DR}}(\pi_e) = \underbrace{\frac{1}{n}\sum_{t=1}^{n} \mathbb{E}_{\mathbf{a} \sim \pi_e(\cdot|u_t)}[\hat{r}(u_t, \mathbf{a})]}_{\text{DM term}} + \underbrace{\frac{1}{n}\sum_{t=1}^{n} \frac{\pi_e(\mathbf{a}_t|u_t)}{\pi_0(\mathbf{a}_t|u_t)} \left(r_t - \hat{r}(u_t, \mathbf{a}_t)\right)}_{\text{IPS residual correction}}$$

where $\hat{r}(u, \mathbf{a})$ is a learned reward model.

**Unpacking the structure:**
- **DM term:** Estimate the value of $\pi_e$ by applying the reward model to hypothetical actions from $\pi_e$. This has low variance but is biased if $\hat{r}$ is misspecified.
- **IPS residual:** The difference $r_t - \hat{r}(u_t, \mathbf{a}_t)$ is the reward **residual** — what the model failed to predict. This is then IPS-weighted to correct for the logging policy. If $\hat{r}$ is good, the residuals are small, and the IPS term has low variance.

### Why "Doubly Robust"

DR is unbiased if **either** (not necessarily both) of the following holds:

**Condition A:** The reward model $\hat{r}$ is correctly specified: $\hat{r}(u, \mathbf{a}) = \mathbb{E}[r | u, \mathbf{a}]$

**Condition B:** The propensity model $\pi_0$ is correctly specified

**Proof of Condition A:** If $\hat{r}$ is correct, then $\mathbb{E}[r_t - \hat{r}(u_t, \mathbf{a}_t)] = 0$, so the IPS residual has zero expectation and the DM term alone gives the correct estimate.

**Proof of Condition B:** If propensities are correct, the IPS term is an unbiased correction. Write:

$$\mathbb{E}[\hat{V}_{\text{DR}}] = V_{\text{DM}}(\pi_e) + \mathbb{E}_{\pi_0}\left[w_t(r_t - \hat{r}_t)\right]$$

$$= V_{\text{DM}}(\pi_e) + \mathbb{E}_{\pi_e}[r - \hat{r}] = V_{\text{DM}}(\pi_e) + V(\pi_e) - V_{\text{DM}}(\pi_e) = V(\pi_e) \checkmark$$

This is the "double robustness" property: you get one free mistake. In practice, you will make both mistakes (neither model is perfectly specified), so DR is not a magic bullet — but it is more robust than either DM or IPS alone.

### Practical Limitations at Scale

1. **The DM term requires evaluating $\hat{r}$ under $\pi_e$'s action distribution.** For ranking over $N$ items, this means marginalizing over $K!$ possible slates — computationally intractable. In practice, this is approximated by sampling from $\pi_e$ or by assuming decomposable rewards.

2. **The reward model $\hat{r}$ must generalize to $\pi_e$'s support.** If $\pi_e$ surfaces items that $\pi_0$ never showed, $\hat{r}$ must extrapolate — often poorly.

3. **Correlated errors amplify bias.** If the reward model error and propensity estimation error are correlated (which they often are, since both depend on item popularity), neither robustness guarantee applies cleanly.

## 3.4 Position-Aware Evaluation: The Position-Based Model

### Derivation of the Position-Based Model (PBM)

The PBM (Craswell et al., 2008) decomposes click probability as:

$$P(C_{u,i,k} = 1) = P(E_k = 1) \cdot P(C_{u,i} = 1 | E_k = 1)$$

where:
- $C_{u,i,k}$: click event for user $u$, item $i$ at position $k$
- $E_k$: examination event at position $k$ (position-dependent, item-independent)
- $P(C_{u,i} = 1 | E_k = 1) = \alpha_{u,i}$: item-user relevance (position-independent)

This yields:
$$P(C_{u,i,k} = 1) = \theta_k \cdot \alpha_{u,i}$$

The key assumption: examination probability $\theta_k$ depends **only on position**, not on item identity or user. This is the PBM's independence assumption, and it is the primary failure mode (see trust bias below).

### Estimating Propensities from Randomization (Intervention Harvesting)

The **propensity** in the PBM context is $\theta_k$ — the examination probability at rank $k$. To estimate it without randomizing your entire system:

**Randomization experiment:** For a fraction of traffic, randomly swap item $i$ between position $k$ and position $k'$ (keeping all other items fixed). The ratio of click rates:

$$\frac{\text{CTR}(i, k)}{\text{CTR}(i, k')} = \frac{\theta_k \cdot \alpha_{u,i}}{\theta_{k'} \cdot \alpha_{u,i}} = \frac{\theta_k}{\theta_{k'}}$$

The $\alpha_{u,i}$ terms cancel, giving a clean estimate of the relative examination probabilities. Normalizing by $\theta_1 = 1$ (or any reference position) gives absolute propensities.

**Intervention harvesting** (Agarwal et al., 2019): Instead of running a dedicated randomization experiment, harvest natural interventions from the log — cases where the same item appeared at different positions across queries. These occur naturally due to personalization diversity or A/B test traffic. This is more efficient but requires controlling for confounders (why did the item appear at different positions? If it's due to different query quality, the confound is real).

### Trust Bias vs. Position Bias

**Position bias:** Users are more likely to *examine* high-ranked items. This affects the probability that the user sees the item at all.

**Trust bias** (also called *presentation bias* or *authority bias*): Users are more likely to *click* high-ranked items given that they examined them, because they infer that the system has judged top-ranked items to be better. Formally:

$$P(C_{u,i,k} = 1 | E_k = 1) = \alpha_{u,i} \cdot \beta_k$$

where $\beta_k$ is a position-dependent trust factor ($\beta_1 > \beta_2 > \ldots$).

The full model with both biases:

$$P(C_{u,i,k} = 1) = \theta_k \cdot \beta_k \cdot \alpha_{u,i}$$

**Why the distinction matters:** If you run a randomization experiment that only isolates $\theta_k$ (by swapping items and measuring click rate changes), but trust bias $\beta_k$ is also present, your propensity estimate captures $\theta_k \cdot \beta_k$ — which is not separable without additional experimental design. Joachims et al. (2017) show that this conflation leads to systematically underestimating the propensity at low positions (trust bias amplifies examination bias).

**Practical implication:** Propensity estimates from observational data should be treated as capturing a combined position+trust effect. Using them to debias training is still valuable but doesn't fully decompose the two mechanisms.

## 3.5 The Replay Method

### How It Works

The Replay method (Li et al., 2011) is an offline evaluation technique for bandit algorithms and recommendation policies that uses **rejection sampling** to simulate a uniform logging policy.

**Algorithm:**
1. Collect a log $\mathcal{D} = \{(u_t, \mathbf{a}_t, r_t)\}_{t=1}^{n}$ under logging policy $\pi_0$
2. For each event $t$ in the log:
   a. Query the evaluation policy $\pi_e$ for its action: $\hat{\mathbf{a}}_t = \pi_e(u_t)$
   b. **Accept** event $t$ if $\hat{\mathbf{a}}_t = \mathbf{a}_t$ (i.e., the evaluation policy agrees with what was actually shown)
   c. **Reject** otherwise
3. Estimate $V(\pi_e) = \frac{\sum_{t : \text{accepted}} r_t}{|\text{accepted}|}$

### The Core Assumption

Replay requires that the logging policy is **uniform over the action space**: $\pi_0(\mathbf{a} | u) = 1/|\mathcal{A}|$ for all $(u, \mathbf{a})$.

Under this assumption, conditional on $\hat{\mathbf{a}}_t = \mathbf{a}_t$ (acceptance), the event is a valid sample from the evaluation policy's action distribution — because the action $\mathbf{a}_t$ was chosen by $\pi_0$ uniformly (not because it was estimated to be good).

**Unbiasedness proof:** Under uniform $\pi_0$:

$$P(\text{accept at time } t) = P(\pi_e(u_t) = \mathbf{a}_t) = \pi_e(\mathbf{a}_t | u_t) \cdot \frac{1}{|\mathcal{A}|} \cdot |\mathcal{A}| = \pi_e(\mathbf{a}_t | u_t)$$

Wait — more precisely: given a uniform logging policy, the probability that event $t$ is accepted is:

$$P(\text{accept}) = \sum_{\mathbf{a}} \pi_0(\mathbf{a}|u_t) \cdot \mathbb{1}[\pi_e(u_t) = \mathbf{a}] = \frac{1}{|\mathcal{A}|}$$

And among accepted events, $r_t$ is an unbiased estimate of $\mathbb{E}[r | u_t, \pi_e(u_t)]$ — which is exactly the policy value.

### When the Assumption Breaks

Production logging policies are **never uniform**. They are sophisticated rankers optimized for user engagement. This means:

1. **Acceptance rate is extremely low** for policies that differ substantially from $\pi_0$ — most candidate actions from $\pi_e$ will never match $\pi_0$'s output, so the accepted subset is tiny and potentially unrepresentative.

2. **The accepted events are biased** — they are the events where $\pi_e$ and $\pi_0$ agree, which may systematically differ from $\pi_e$'s full distribution.

3. **Sample efficiency collapses** when $\pi_0$ is deterministic — a deterministic logging policy accepts only events where $\pi_e$ makes exactly the same decision, which shrinks the evaluation set to near-zero.

**Practical use:** Replay is appropriate only when: (a) you have a stochastic exploration policy in production (e.g., epsilon-greedy or Thompson sampling logged data), (b) the exploration rate is high enough to provide sufficient coverage of $\pi_e$'s action space, and (c) the evaluation policy is close enough to $\pi_0$ that acceptance rate is non-trivial. In practice, this makes Replay most useful for evaluating bandit arms in recommendation, not for evaluating full ranking systems.

---

## Level 3 Study Aids

### Concept Check

**Q1.** You implement IPS-weighted NDCG to debias your offline evaluation. You notice that for 0.1% of training examples, the importance weight $w = \pi_e / \pi_0$ exceeds 1000. These examples account for 40% of the total weighted reward. What does this tell you about the quality of your IPS estimate, and what would you do about it? Quantify the bias-variance tradeoff for a clipping threshold of $M = 100$.

**Q2.** You have access to a production system with a deterministic ranker ($\pi_0$ always produces the same ranked list for the same query). You want to evaluate a new model using the Replay method. Explain precisely why the Replay method will fail in this setting and propose an alternative experimental design that would give you an unbiased evaluation.

**Q3.** The Doubly Robust estimator is guaranteed to be unbiased if either the reward model OR the propensity model is correct. In a real production system, both are wrong. Under what specific conditions would you expect DR to perform *worse* than simple IPS? Construct a concrete scenario.

---

### Common Misconceptions

**Misconception 1: "IPS debiases the evaluation — therefore I can trust my offline metric."**

IPS debiases the evaluation **under the assumption that the propensity model is correct** and that $\pi_e$'s support is contained in $\pi_0$'s support. Both assumptions are violated in practice. IPS with misspecified propensities can produce an estimate that is more biased than naive evaluation, especially when the propensity errors are correlated with item quality (which they typically are — popular items have more accurate propensity estimates than tail items).

**Misconception 2: "SNIPS is strictly better than IPS."**

SNIPS has lower variance but is biased in finite samples. For small datasets or when statistical guarantees require unbiasedness, IPS is preferable. The choice depends on the data regime: small $n$ → favor IPS for unbiasedness guarantee; large $n$ → favor SNIPS for variance reduction.

**Misconception 3: "Position-based debiasing is the same as removing position as a feature."**

Removing position as a feature from a ranking model prevents the model from *exploiting* position effects in training — but it does not correct the position bias in the *labels*. Your click labels are still corrupted by $\theta_k$ regardless of whether position is a feature. Position-based debiasing (PBM + IPS) corrects the label-level bias, not the feature-level bias.

---

### Connections

- **← Level 1:** Every debiasing technique in this level targets one of the three biases described in Level 1. IPS addresses selection bias (reweighting logged samples by their exposure probability). Position-based models address position bias. DR combines both with a reward model to reduce variance. None of them fully addresses popularity bias without additional constraints on the reward model.

- **→ Level 4:** Debiasing techniques improve the quality of offline evaluation, but they do not close the offline-online gap. Even with perfect IPS (correct propensities, no clipping), the policy gap $\delta(\pi_0, \pi_e)$ means your offline estimate has high uncertainty for genuinely novel rankers. Level 4 addresses this gap directly.

- **→ Level 5:** The papers by Joachims et al. (2017) and Schnabel et al. (2016) in Level 5 are the foundational derivations that this level summarizes. Reading them after this level, you should be able to follow every mathematical step.

---

<a id='level-4'></a>
# LEVEL 4: OFFLINE-ONLINE CORRELATION AND METRIC DESIGN

---

## 4.1 The Offline-Online Gap

### What It Is

The **offline-online gap** is the phenomenon where a model achieving higher offline metric (e.g., NDCG) sometimes produces no measurable improvement in live A/B test metrics (CTR, conversion, long-session engagement). It is one of the most practically consequential problems in production ML.

Formally, let $\Delta_{\text{off}}(A, B) = \text{NDCG}(B) - \text{NDCG}(A)$ be the offline lift of model B over model A. Let $\Delta_{\text{on}}(A, B)$ be the online lift in an A/B test. The gap is:

$$\text{gap}(A, B) = \mathbb{E}[\Delta_{\text{on}}] - f(\Delta_{\text{off}})$$

for some calibration function $f$. The gap problem is that $f$ is poorly calibrated (or non-monotone) in practice.

### Mechanism 1: Distributional Shift

Your offline evaluation set is a historical sample from a past traffic distribution. The live traffic at A/B test time may differ in:
- **Query distribution:** Seasonality, trending topics, geographic shifts
- **User distribution:** New users, churned users, user population composition
- **Item distribution:** New items not in the training set, item quality drift
- **Context distribution:** Device type changes, UI changes, time-of-day effects

If the evaluation set $\mathcal{D}_{\text{eval}}$ is drawn from distribution $P_{\text{hist}}$ and live traffic is from $P_{\text{live}}$, then:

$$\hat{\Delta}_{\text{off}} \approx \mathbb{E}_{P_{\text{hist}}}[\text{lift}] \neq \mathbb{E}_{P_{\text{live}}}[\text{lift}] = \Delta_{\text{on}}$$

whenever $P_{\text{hist}} \neq P_{\text{live}}$.

**Mitigation:** Construct evaluation sets that match the expected live traffic distribution as closely as possible: use recent data (within the last 2–4 weeks), segment by query type, and weight evaluation examples by live traffic frequency.

### Mechanism 2: The Policy Gap

As established in Level 1, a new ranker $\pi_e$ surfaces items that $\pi_0$ (the logging policy) never showed. These items have no logged outcomes and cannot be evaluated from logs. The portion of $\pi_e$'s output that is genuinely novel is effectively evaluated by extrapolation from the reward model — which is the weakest part of the offline evaluation pipeline.

Concretely: if your new model uses better semantic understanding to retrieve previously unseen long-tail items, those items will have few or no clicks in your evaluation set. The model will be penalized for retrieving them (they appear as zero-relevance items in the evaluation), even though they may be highly relevant to users.

### Mechanism 3: Novelty Effects and User Adaptation

**Novelty effect:** In A/B tests, the treatment group sometimes shows temporarily inflated CTR simply because users notice something different (a new UI, different items) and explore it. This is a short-term artifact, not a long-term quality improvement.

**Adaptation effect:** The converse — users initially perform worse with a new system because they need to learn new interaction patterns. A recommendation system with a different content mix may see short-term CTR drops even if long-term satisfaction increases.

Neither of these effects is captured in offline evaluation. Offline metrics implicitly assume that user behavior is a static function of the recommendation — which is false. Users adapt, and their preferences shift in response to what the system recommends over time.

## 4.2 Building Metrics That Predict Online Outcomes

### Calibration Between Offline and Online Delta

The goal is to estimate the **calibration function** $f$ such that $\mathbb{E}[\Delta_{\text{on}}] \approx f(\Delta_{\text{off}})$. This requires historical data from past model launches where both offline and online lifts were measured.

**Procedure:**
1. Collect $M$ past model launches with measured $(\Delta_{\text{off}}^{(m)}, \Delta_{\text{on}}^{(m)})$
2. Fit a calibration model: $f(\delta) = \hat{\alpha} + \hat{\beta} \cdot \delta$ (linear as a baseline, nonlinear if warranted)
3. Evaluate predictive $R^2$: if $R^2$ is low, the offline metric is not predictive of online outcomes — consider changing metrics

**Key diagnostic:** Compute the **Spearman rank correlation** between $\Delta_{\text{off}}$ and $\Delta_{\text{on}}$ across your historical launches. If $\rho < 0.5$, your offline metric is not reliably predictive. If $\rho < 0$ (offline positive, online negative), the metric is **anti-correlated** and should be abandoned immediately.

### Minimum Detectable Effect

The **Minimum Detectable Effect (MDE)** for an offline metric is the smallest $\Delta_{\text{off}}$ that reliably predicts a positive $\Delta_{\text{on}}$:

$$\text{MDE}_{\text{off}} = \frac{\text{MDE}_{\text{on}}}{\hat{\beta}}$$

where $\hat{\beta}$ is the calibration slope and $\text{MDE}_{\text{on}}$ is the minimum online effect detectable by your A/B test infrastructure (typically 0.5%–1% CTR lift for large platforms).

If your calibration shows that a 1% offline NDCG lift corresponds to a 0.1% online CTR lift, and your A/B infrastructure can detect 0.5% CTR lifts with 80% power, then you need at least 5% offline NDCG lift to have reasonable confidence that the model will pass an A/B test.

### Industry Thresholds

**Netflix (artwork personalization):** Netflix reported that offline metric improvements needed to exceed approximately 1–2% to predict reliable online engagement lift (measured via play rate). Below this threshold, models were not A/B tested due to noise. Their offline metric of choice was a proxy derived from implicit positive/negative signal from image click-through, calibrated against completed-play probability.

**Airbnb (search):** In their published work (Haldar et al., 2019), Airbnb found that offline NDCG improvements did not reliably predict booking rate improvements. They moved toward a composite offline metric incorporating booking probability predictions directly, calibrated against historical experiments. Their threshold for initiating an A/B test was typically a >0.5% improvement in their composite metric.

**Pinterest (Pinnability):** Pinterest developed a metric called "Pinnability" that combines engagement signals (saves, closeups, link clicks) with personalization signals, calibrated against their primary online KPI (long-term engagement). They found that the correlation between single-signal offline metrics (pure CTR or pure NDCG) and online KPIs was low; Pinnability was more predictive because it modeled the composition of engagement types.

**General principle:** No industry uses a single raw academic metric (NDCG, MAP) as the sole gate for A/B test decisions. The practice is to build composite metrics calibrated against historical online data, with thresholds set based on the A/B test's statistical power.

## 4.3 Interleaving as an Alternative to Offline Eval

### Team-Draft Interleaving

Interleaving (Radlinski & Craswell, 2013) is a technique that produces a single merged ranking from two competing policies and uses implicit user feedback to determine which policy is preferred — without a full A/B test.

**Team-Draft Interleaving Algorithm:**
1. Both ranking systems A and B produce ranked lists for a query
2. Construct an interleaved list $L$ using a team-draft process:
   - Flip a coin to decide which system picks first (say A)
   - A adds its top-ranked unselected item to $L$
   - B adds its top-ranked unselected item that is not already in $L$
   - Alternate until $K$ items are selected
3. Show $L$ to the user and observe clicks $C$
4. Credit each click to the system that contributed that item: $\text{win}(A) = |C \cap L_A|$, $\text{win}(B) = |C \cap L_B|$
5. Aggregate $\Delta_{\text{AB}} = \text{win}(A) - \text{win}(B)$ across all queries

**Statistical properties:**
- Unbiased: the team-draft process ensures that items from A and B are placed at positions with equal probability in expectation, so position bias cancels.
- Zero-sum: $\text{win}(A) + \text{win}(B) = |C|$
- The test statistic is a sign test or binomial test on $\Delta_{\text{AB}}$

### Why Interleaving Has Higher Sensitivity Than A/B Tests

In a standard A/B test, user variance is a major source of noise: different users have different click rates, session lengths, and engagement levels. The difference between policies A and B is estimated *across* users, which includes all between-user variability.

In interleaving, both policies compete **within the same session for the same user**. The between-user variability cancels out because every user serves as their own control. The effect is:

$$\text{Var}[\Delta_{\text{interleave}}] \ll \text{Var}[\Delta_{\text{A/B}}]$$

Empirically, interleaving is reported to require 100x–1000x less traffic than A/B tests to detect the same effect size (Chapelle et al., 2012). This makes interleaving the method of choice when you need to rapidly compare many ranking systems and live traffic is expensive.

### When Interleaving Is Appropriate vs. Inappropriate

**Appropriate:**
- Comparing two ranking systems on the same item set (ranker comparison)
- Situations where the result list is the primary user interface element
- Settings with low traffic where A/B tests would take too long

**Inappropriate:**
- Comparing retrieval systems (item sets differ, so interleaving cannot safely merge them)
- UI changes or non-ranking changes (interleaving cannot attribute outcomes to non-item factors)
- Personalization changes that affect item pool diversity — if A shows a diverse set and B shows a narrow set, interleaving conflates the quality of ranking with the quality of item selection
- Metrics beyond click (conversion, purchase, long-term retention) — interleaving is optimized for session-level engagement signals

## 4.4 Counterfactual Learning to Rank (CLTR)

### Unbiased LTR vs. Standard LTR

**Standard LTR** trains a ranking model to maximize a ranking metric (e.g., NDCG) computed from click labels directly. As established, click labels are confounded by position bias. Standard LTR therefore optimizes a **biased objective** — it learns to reproduce the biases in the training data.

**Unbiased LTR** uses IPS-weighted variants of ranking metrics as the training objective:

$$\mathcal{L}_{\text{unbiased}}(f) = -\sum_{(u, q, i)} \frac{c_{u,q,i}}{\hat{\theta}_{k_{u,q,i}}} \cdot \delta(\sigma_f, i)$$

where:
- $c_{u,q,i}$: click indicator for item $i$ shown to user $u$ for query $q$
- $\hat{\theta}_{k_{u,q,i}}$: estimated examination probability at the position where item $i$ was shown
- $\delta(\sigma_f, i)$: the ranking metric contribution of item $i$ under ranking $\sigma_f$ produced by model $f$

The IPS weight $1/\hat{\theta}_k$ upweights items shown at low positions (high IPS weight) and downweights items shown at high positions (low IPS weight), correcting for the examination probability confound.

### The Dual Learning Algorithm (DLA)

DLA (Ai et al., 2018) addresses the chicken-and-egg problem: to learn an unbiased ranker, you need propensity estimates; to estimate propensities, you need a ranker. DLA learns both simultaneously.

**Setup:** Two models:
- $f(u, q, i)$: ranker — estimates item relevance
- $g(k)$: propensity estimator — estimates examination probability at rank $k$

**DLA loss function:**

$$\mathcal{L}_{\text{DLA}} = -\sum_{(u,q,i,k)} c_{u,q,i} \cdot \left[\log \sigma\left(\frac{f(u, q, i)}{g(k)}\right) + \log \sigma\left(\frac{g(k)}{f(u, q, i)}\right)\right]$$

The loss is symmetric: the ranker $f$ is trained with $g$ as the propensity correction, and $g$ is trained with $f$ as the relevance correction. The key insight is that clicks are explained by the product $f \cdot g$, so if either model is good, the other can be improved.

**Convergence:** DLA converges to a Nash equilibrium where both $f$ and $g$ are locally consistent. It requires careful initialization (starting from a reasonable relevance model and propensity model) to avoid bad local optima.

### Unbiased LambdaMART (Hu et al., 2019)

LambdaMART is the standard gradient-boosted LTR algorithm used in production at most search companies. **Unbiased LambdaMART** (Hu et al., 2019) modifies the lambda weights to account for position bias:

Standard LambdaMART computes pairwise gradients:

$$\lambda_{ij} = \frac{-\partial \text{NDCG}}{\partial s_i - \partial s_j} \cdot \mathbb{1}[\text{rel}(i) > \text{rel}(j)]$$

Unbiased LambdaMART modifies these gradients with propensity weights:

$$\lambda_{ij}^{\text{unbiased}} = \frac{\lambda_{ij}}{\hat{\theta}_{k_i} \cdot \hat{\theta}_{k_j}}$$

dividing each pairwise gradient by the product of examination probabilities at the positions where the two items were shown. This reweights the training gradient to simulate what the loss would be if items were shown uniformly.

**Key insight:** Hu et al. show that standard LambdaMART on click data converges to a biased solution that ranks items proportional to $\text{rel}(i) \cdot \theta_k$ rather than $\text{rel}(i)$. Unbiased LambdaMART corrects this and converges to a solution proportional to $\text{rel}(i)$ alone.

---

## Level 4 Study Aids

### Concept Check

**Q1.** Your team has launched 20 models over the past year with measured offline NDCG lifts and online CTR lifts. You compute a Spearman rank correlation of $\rho = 0.2$ between offline and online lifts. Your manager asks: "Should we trust our offline metric?" Give a principled answer, including: (a) what $\rho = 0.2$ implies about metric predictivity, (b) what alternative metrics you would investigate, and (c) what changes to the evaluation methodology might improve correlation.

**Q2.** You want to compare two retrieval systems: System A uses BM25 and System B uses dense neural retrieval (DPR). Can you use interleaving to compare them? If not, why not, and what alternative evaluation methodology would you use?

**Q3.** DLA jointly learns a ranker and propensity estimator. Suppose the ranker converges to a solution that perfectly predicts clicks ($f(u, q, i) \propto c_{u,q,i}$). What does the propensity estimator $g$ converge to in this case, and is it a valid propensity estimate? What does this tell you about the initialization requirements for DLA?

---

### Common Misconceptions

**Misconception 1: "If our offline metric correlates with online at $\rho = 0.6$, it's reliable."**

Correlation $\rho = 0.6$ means 64% of variance in online lift is explained by offline lift — which also means 36% of variance is noise. For individual model launch decisions, this is not reliable enough. You need to understand the **distribution** of the unexplained variance: if residuals are small relative to your MDE, $\rho = 0.6$ may be acceptable; if residuals are large, you will launch models that fail A/B tests at an unacceptable rate.

**Misconception 2: "Interleaving is always better than A/B testing."**

Interleaving is more statistically efficient for comparing *rankers* but measures *click preference*, not long-term engagement, conversion, or retention. Many important online metrics (user retention, subscription renewal, lifetime value) cannot be captured by interleaving. A/B tests measure the full treatment effect including all downstream metrics; interleaving measures only session-level click preference.

**Misconception 3: "Unbiased LTR means the ranking model is unbiased."**

"Unbiased LTR" means the learning objective is an unbiased estimate of the ranking metric under a specific propensity model. If the propensity model is wrong, the learning is still biased. If the data coverage is insufficient (selection bias from unobserved items), the model is still biased. The "unbiased" label refers to the statistical property of the estimator under stated assumptions, not to the real-world quality of the resulting ranker.

---

### Connections

- **← Level 3:** CLTR is the training-time application of the evaluation-time debiasing techniques from Level 3. IPS is used in evaluation (Level 3) and as a gradient correction in training (Level 4). The propensity estimation problem is shared: the same position-based model propensities estimated in Level 3 feed directly into the training objective of Unbiased LambdaMART.

- **← Level 1:** The policy gap problem (Level 1) directly bounds how much the offline-online gap can be closed. Even with perfect offline evaluation methodology (IPS, SNIPS, DR), the offline estimate becomes unreliable as $\delta(\pi_0, \pi_e)$ grows. This is why interleaving (which does not rely on logged data) becomes important for highly novel ranking systems.

- **→ Level 5:** The industry practices in Level 5 are specific implementations of the principles in this level. Netflix's proxy metric work, Airbnb's composite metric, and Pinterest's Pinnability are all concrete attempts to solve the calibration problem described in Section 4.2.

---

<a id='level-5'></a>
# LEVEL 5: INDUSTRY PRACTICE AND CANONICAL PAPERS

---

## 5.1 Canonical Papers

### Joachims et al. (2017) — "Unbiased Learning-to-Rank with Biased Feedback"
*WSDM 2017*

**Core problem:** Standard LTR trains on click data that is confounded by position bias. A system trained on this data will learn to reproduce position bias rather than learning true relevance.

**Approach:** Derive an IPS-based propensity-weighted ranking objective. Show that click-based LTR without debiasing converges to the wrong solution. Use randomization experiments to estimate examination propensities $\theta_k$.

**Key insight:** The propensity score at rank $k$ can be estimated by random interventions (swapping items across positions) without full randomization of the system. The key equation:

$$\hat{\Delta}_{\text{NDCG}}^{\text{IPS}}(f) = \sum_{(u, i, k)} \frac{c_{u,i,k}}{\theta_k} \cdot \Delta_{\text{NDCG}}(f, u, i)$$

They prove this is an unbiased estimator of the true NDCG-based gradient under the PBM assumption.

**Practitioner takeaway:** (1) Always collect randomization data alongside production traffic — even 1% of traffic with random position swaps gives you propensity estimates for all positions. (2) The debiased training objective is a drop-in replacement for standard LTR objectives — the architecture does not change. (3) The paper provides the first rigorous proof that click-based LTR is convergently biased, not just noisy.

---

### Schnabel et al. (2016) — "Recommendations as Treatments: Debiasing Learning and Evaluation"
*ICML 2016*

**Core problem:** Evaluating and training recommender systems from observational (logged) data, where users only rate or interact with items they were exposed to. The MNAR structure of this data causes standard matrix factorization and evaluation metrics to be biased.

**Approach:** Adopt the causal inference framework of **Rubin's Potential Outcomes**. Frame recommendations as treatments: item $i$ shown to user $u$ is a treatment, and the potential outcome is $Y_{u,i}(1)$ (shown) or $Y_{u,i}(0)$ (not shown). The propensity $P(M_{u,i} = 1)$ is the probability that item $i$ is shown to user $u$ under the logging policy.

**Key insight:** Both the *training* objective and the *evaluation* metric should be propensity-weighted. Standard MSE on observed ratings:

$$\hat{\mathcal{L}}_{\text{naive}} = \frac{1}{|\mathcal{D}_{\text{obs}}|} \sum_{(u,i) \in \mathcal{D}_{\text{obs}}} (\hat{r}_{u,i} - r_{u,i})^2$$

is a biased estimator of the true MSE over all $(u, i)$ pairs. The IPS-corrected estimator:

$$\hat{\mathcal{L}}_{\text{IPS}} = \frac{1}{|\mathcal{U}||\mathcal{I}|} \sum_{(u,i) \in \mathcal{D}_{\text{obs}}} \frac{(\hat{r}_{u,i} - r_{u,i})^2}{P(M_{u,i} = 1)}$$

is unbiased.

They also introduce the concept of **exposure propensity estimation** from logged data using a propensity model trained on item features, user features, and interaction density.

**Practitioner takeaway:** (1) The key equations for IPS-corrected training and evaluation are clean and implementable — this paper gives the recipe. (2) Propensity estimation from item/user features (without a randomization experiment) is a viable but noisier alternative; validate propensity quality with calibration plots. (3) This paper introduced the systematic framing of recommendation as a causal inference problem, which is now standard language in the field.

---

### Wang et al. (2018) — "Position-Bias Estimation for Unbiased Learning to Rank in Personal Search"
*WSDM 2018, Google*

**Core problem:** Estimating position bias propensities for personal search (Gmail search, Google Drive) without contaminating the user experience with randomization experiments.

**Approach:** **Intervention harvesting** — use natural variation in results across users and queries to estimate relative examination probabilities. Specifically, they exploit the fact that in personal search, the same document may appear at different positions for different users (due to personalization), providing natural counterfactual observations.

**Key insight:** The EM algorithm over the PBM can be used to jointly estimate relevance parameters $\alpha_{u,i}$ and examination parameters $\theta_k$ from click data alone, without any randomization. The EM update alternates:

**E-step:** $\hat{\theta}_k \propto \frac{\sum_{(u,i,k)} c_{u,i,k}}{\sum_{(u,i,k)} \hat{\alpha}_{u,i}}$

**M-step:** $\hat{\alpha}_{u,i} \propto \frac{\sum_k c_{u,i,k}}{\sum_k \hat{\theta}_k}$

This is identifiable only up to a multiplicative constant — one reference position must be fixed (e.g., $\theta_1 = 1$).

**Practitioner takeaway:** (1) You do not need a randomization experiment to estimate position propensities — the EM approach works from observational data alone. (2) The EM estimate conflates position bias and trust bias (Section 3.4); be aware of this. (3) The propensity estimates degrade for rare (u, i) pairs — regularize the $\hat{\alpha}_{u,i}$ estimates toward a population mean.

---

### Swaminathan & Joachims (2015) — "Counterfactual Risk Minimization: Learning from Logged Bandit Feedback"
*ICML 2015*

**Core problem:** How do you train a policy from historical bandit feedback in a way that is statistically guaranteed to improve over the logging policy?

**Approach:** Formulate the problem as **Counterfactual Risk Minimization (CRM)**. The objective is to minimize an upper bound on the true risk under the evaluation policy, using a generalization bound derived from the Rademacher complexity of the policy class.

The CRM objective:

$$\hat{\pi}_{\text{CRM}} = \arg\min_{\pi \in \Pi} \underbrace{\hat{V}_{\text{IPS}}(\pi)}_{\text{SNIPS estimate}} + \lambda \cdot \underbrace{\sqrt{\frac{\text{Var}[w(\pi) \cdot r]}{n}}}_{\text{variance penalty}}$$

The variance penalty $\sqrt{\text{Var}/n}$ is directly estimated from the data and acts as a regularizer that penalizes high-variance (high-importance-weight) policies.

**Key insight:** SNIPS + variance regularization is the correct objective for learning from bandit data — not raw IPS. The variance penalty automatically penalizes policies that deviate too far from $\pi_0$ (which would produce high importance weights and high variance estimates), encouraging the learned policy to stay in the support of the logging data.

**Practitioner takeaway:** (1) CRM is the formal justification for the "don't change too much from your current policy" heuristic. (2) The variance penalty $\lambda$ controls the policy shift — tune it based on how much exploration budget you have. (3) This paper bridges bandit theory and practical recommendation learning, providing the theoretical foundation for SNIPS-based training objectives.

---

### Saito & Joachims (2021) — "Doubly Robust Estimator for Ranking Metrics with Post-Click Conversions"
*KDD 2021*

**Core problem:** Clicks are biased (position bias, trust bias) and conversions are doubly biased (you only observe conversion for clicked items, so conversion data has both the position bias of clicks and the selection bias of clicking itself).

**Approach:** Derive a doubly robust estimator for ranking metrics that use conversion (purchase, signup, etc.) as the relevance signal, accounting for the **two-stage selection process** (item must be shown → item must be clicked → conversion is observed).

The DR estimator for conversion-based ranking:

$$\hat{V}_{\text{DR-conv}}(\pi_e) = \hat{V}_{\text{DM}} + \frac{1}{n}\sum_t \frac{c_t}{\hat{\theta}_{k_t}} \cdot \frac{\text{conv}_t - \hat{\mu}(u_t, i_t)}{\hat{\rho}(u_t, i_t)}$$

where:
- $c_t/\hat{\theta}_{k_t}$: IPS correction for position bias on clicks
- $\text{conv}_t - \hat{\mu}(u_t, i_t)$: residual from conversion model
- $\hat{\rho}(u_t, i_t)$: propensity of conversion given click (the second-stage selection)

**Key insight:** For post-click metrics, you need two propensity corrections: one for the examination-click path and one for the click-conversion path. A single IPS correction is insufficient.

**Practitioner takeaway:** (1) If you are optimizing for conversion (not clicks), use the two-stage DR estimator. (2) Conversion propensity $\hat{\rho}$ can be estimated from historical click-to-conversion rates segmented by item category, price, and user engagement history. (3) This paper is highly relevant for e-commerce search and marketplace recommendation where purchase is the primary signal.

## 5.2 Industry Practice: Blog Posts and Case Studies

### Netflix: Artwork Personalization and Offline Proxy Metrics

**Source:** Chandrashekar et al. (2017), "Artwork Personalization at Netflix"; multiple Netflix Tech Blog posts on A/B testing and metric design.

**Core problem:** Netflix personalizes the artwork (thumbnail images) shown for each title to each user. They needed an offline evaluation method to predict which artwork would perform best in an online A/B test.

**Approach:** Netflix built a **proxy metric** based on image impression-to-play rate, controlling for position effects and user engagement state. The key insight was that click-through rate on artwork was a poor proxy for what they actually cared about (completed plays, long-term engagement). They developed an engagement-adjusted metric that weighted clicks by downstream completion probability.

**Offline to online calibration:** Netflix reported that they needed consistent offline metric improvements above a threshold (approximately 1–2% in their engagement-adjusted metric) before initiating an A/B test. Below this threshold, the noise in the offline estimate made A/B test outcomes unreliable.

**Key insight for practitioners:** The decision to use a composite engagement metric (not raw CTR) came from empirical evidence that CTR-optimized artwork drove short-term clicks but users abandoned videos that didn't match the artwork's implied quality. This is a textbook example of Goodhart's Law — optimizing CTR misaligned with the true objective (content consumption).

**What to take away:** (1) Build offline metrics that are composites of multiple signals weighted by downstream value, not single-signal proxies. (2) Validate offline metric choices empirically by computing correlation with historical A/B test outcomes. (3) Set a minimum offline improvement threshold before committing to an A/B test.

---

### Airbnb: Applying Deep Learning to Search

**Source:** Haldar et al. (2019), "Applying Deep Learning to Airbnb Search" (KDD 2019)

**Core problem:** Airbnb's search ranking determines which listings are shown to guests. The primary business metric is booking rate (not clicks or page views). Offline evaluation needed to predict booking rate lift, not just engagement.

**Approach:** Airbnb found that offline NDCG (computed against click labels) was poorly predictive of booking rate. They developed an **offline booking-probability-weighted NDCG** that used a booking probability model trained on historical booking data as the relevance score, rather than raw clicks.

$$\text{NDCG}_{\text{Airbnb}}@K = \frac{\sum_{k=1}^{K} \frac{2^{\hat{p}_{\text{book}}(i_k | u, q)} - 1}{\log_2(k+1)}}{\text{IDCG}@K}$$

where $\hat{p}_{\text{book}}(i | u, q)$ is the predicted booking probability for listing $i$ for user $u$ on query $q$.

**Key challenge:** Booking probability models are trained on logged data with all the biases of Level 1. They handled this with: (a) careful feature engineering to separate listing quality from position effects, (b) log-scaling booking count features to reduce popularity bias, (c) temporal holdout evaluation (evaluate on future bookings, not random splits).

**What to take away:** (1) Replace click-based relevance scores with predicted downstream business metric scores in your NDCG computation. (2) Use temporal holdout (train on past, evaluate on future) rather than random splits to reduce distribution shift in offline eval. (3) Feature engineering for debiasing (removing position, logged popularity) is as important as algorithmic debiasing.

---

### Pinterest: Pinnability

**Source:** Pinterest Engineering Blog (multiple posts on Pinnability and homefeed ranking)

**Core problem:** Pinterest's homefeed shows image-based content (Pins). Engagement is multi-modal: users can close-up (examine), save (strong positive signal), click through to the source, or ignore. The system needed a single offline score that captured this engagement hierarchy.

**Approach:** Pinterest developed **Pinnability** — a composite engagement score that combines multiple signals with learned weights:

$$\text{Pinnability}(u, i) = \sum_k w_k \cdot \hat{p}_k(u, i)$$

where $w_k$ are the weights for engagement type $k$ (close-up, save, click, share) and $\hat{p}_k$ are predicted probabilities for each engagement type. The weights $w_k$ were calibrated against long-term user engagement metrics via historical A/B test data.

**Key insight:** They found that close-ups (image enlargements) are a leading indicator of saves and long-term engagement, while raw clicks correlate with short-term satisfaction but not long-term retention. The composite score weighted saves and close-ups more heavily than clicks.

**Offline evaluation methodology:** Pinterest evaluates Pinnability@K using held-out interaction data, computing the score on future interactions not used in training. They also monitor diversity metrics (category coverage) to prevent recommendation collapse.

**What to take away:** (1) Design relevance scores that are hierarchies of engagement signals weighted by long-term business value. (2) Use leading indicators (early engagement signals that predict delayed outcomes) to reduce evaluation latency. (3) The weights in a composite score must be calibrated against actual online outcomes — they should not be set by intuition.

---

### LinkedIn: Metric Sensitivity and A/B Testing

**Source:** LinkedIn Engineering Blog: "The ABCs of A/B Testing" and associated publications on metric design and online controlled experiments.

**Core problem:** LinkedIn's feed, job recommendations, and search all need offline metrics that are sensitive enough to detect small quality improvements before committing to expensive A/B tests.

**Approach:** LinkedIn invested heavily in **metric sensitivity analysis** — the process of quantifying how much an offline metric changes in response to a known quality improvement. They define sensitivity as:

$$\text{Sensitivity}(m) = \frac{\mathbb{E}[m(\text{good model})] - \mathbb{E}[m(\text{base model})]}{\text{Std}[m(\text{base model})]}$$

A metric with high sensitivity can detect the same quality difference with fewer queries/users — directly affecting how quickly models can be validated.

**Key finding:** LinkedIn found that metrics computed at smaller K (e.g., NDCG@3 vs. NDCG@20) were more sensitive to ranking quality differences at the top of the list — which is where user attention is concentrated. They also found that **pairwise metrics** (comparing the ranking of clicked vs. not-clicked item pairs) were more sensitive than pointwise metrics (absolute DCG/NDCG) for their use case.

**Metric guardrailing:** LinkedIn uses a system of **guardrail metrics** — metrics that must not decrease even when the primary optimization metric improves. Common guardrails: spam rate, user report rate, session abandonment rate. This implements the multi-objective dashboard approach to avoid Goodhart's Law.

**What to take away:** (1) Measure the sensitivity of your offline metric — not just its correlation with online outcomes, but how much it changes per unit of actual quality difference. (2) Pairwise metrics are often more sensitive than pointwise metrics for ranking tasks. (3) Guardrail metrics are operationally critical — they prevent optimizing one dimension at the cost of silent degradation in others.

---

### DoorDash / Instacart: Marketplace Search Evaluation

**Source:** DoorDash Engineering Blog: "Personalizing the DoorDash Consumer Experience"; Instacart Tech Blog: "How Instacart Delivers on Search Relevance"

**Core problem:** Marketplace search (delivery apps, grocery delivery) has a **supply-side constraint**: not all items are always available. A search for "pizza" may return items that are out of stock, unavailable from the user's assigned store, or have a delivery time outside the user's acceptable window. Standard search evaluation ignores supply-side constraints.

**DoorDash approach:** DoorDash evaluates search quality using a **fulfillment-adjusted** metric: an item that is clicked but cannot be fulfilled (unavailable, out of area) is penalized in the evaluation even if it received a click. Their relevance score:

$$\text{rel}(u, i, t) = \mathbb{1}[\text{click}(u, i, t)] \cdot \mathbb{1}[\text{available}(i, t)] \cdot \mathbb{1}[\text{delivered}(u, i, t)]$$

This three-way interaction ensures that the metric reflects end-to-end user experience, not just intent.

**Instacart approach:** Instacart faces an extreme form of the selection bias problem: the item catalog changes daily (products go in and out of stock), so historical click data is from a different item set than the current catalog. Their evaluation uses **catalog-aware evaluation**: relevance labels are only computed for items currently available in the user's assigned store, filtering out stale signal from discontinued products.

**Key insight for practitioners:** In marketplace settings, evaluation must account for supply constraints, availability dynamics, and fulfillment success — not just relevance and ranking quality. A standard NDCG computed against historical clicks will overestimate the quality of systems that recommend currently unavailable items.

---

## Level 5 Study Aids

### Concept Check

**Q1.** Netflix uses an engagement-adjusted metric calibrated against completed plays. You want to adapt this approach to a podcast recommendation system. What engagement signals would you choose, how would you weight them, and how would you calibrate the weights against your online KPI (which is monthly listening hours)? Be specific about the data you would need and the potential biases in each signal.

**Q2.** Airbnb uses booking-probability-weighted NDCG as their offline metric. They observe that their new deep learning model achieves +3% in this metric versus their previous model. Their historical calibration shows that a +1% offline lift corresponds to approximately +0.3% booking rate online, with $\pm 0.4\%$ uncertainty. Should they A/B test this model? Compute the expected online lift and its confidence interval. What is the probability that the A/B test will be statistically significant (assume their A/B test has 80% power to detect a 0.5% booking rate lift)?

**Q3.** Schnabel et al. (2016) propose propensity-weighted evaluation for recommendation. A colleague argues: "Our system has 500M users and 50M items. The propensity of any specific (user, item) pair is essentially 0 — we can't use IPS." Construct a precise technical response that identifies what the colleague is getting wrong about how propensity models work in practice and proposes a tractable propensity estimation approach for this scale.

---

### Common Misconceptions

**Misconception 1: "Industry companies optimize offline metrics directly."**

Industry companies optimize **online metrics** (CTR, conversion, retention, revenue). Offline metrics are used to filter which models are worth A/B testing. The offline metric is a screening tool, not a training objective or final performance measure. Models that pass offline screening are A/B tested; models that win A/B tests are launched. This two-stage process is standard across Netflix, Airbnb, LinkedIn, and every other major platform.

**Misconception 2: "If the paper's method works on benchmarks, it will work in production."**

Academic benchmarks (MovieLens, MS MARCO, Yahoo LTR) have clean relevance labels, complete interaction data, and known query distributions. Production data has MNAR structure, popularity skew, temporal drift, supply constraints, and user diversity that academic benchmarks do not capture. Every method described in Level 5's papers has been found to require significant adaptation for production use. The gap between benchmark performance and production performance is a major theme in industry ML.

**Misconception 3: "The IPS estimator is the state of the art."**

IPS is the foundational tool, not the final word. Doubly robust estimators dominate when reward models are available. DLA and unbiased LambdaMART are better for end-to-end training. Interleaving is more efficient for pairwise model comparison. The choice depends on your specific data regime, the gap between logging and evaluation policies, and whether you need an evaluation or training method. There is no universally best technique.

---

### Connections (Synthesis Across All Levels)

This final connections section traces the complete arc of the curriculum:

**The fundamental thread:** Every concept in this guide is a response to the core problem of Level 1 — you cannot evaluate a recommendation system like a standard supervised learning problem because the data-generating process is controlled by the logging policy, not by nature.

**The bias thread:**
- Level 1 identifies the three biases (position, popularity, selection)
- Level 2 shows how these biases corrupt every standard metric
- Level 3 provides mathematical tools to correct for these biases in evaluation
- Level 4 shows that even corrected offline evaluation has fundamental limits (the policy gap)
- Level 5 shows how industry resolves this by calibrating offline metrics against online experiments

**The estimator thread:**
- IPS (Level 3) → SNIPS (Level 3) → DR (Level 3): Each estimator builds on the previous, trading bias for variance in increasingly sophisticated ways
- IPS for evaluation (Level 3) → IPS for training (Level 4 CLTR): The same mathematical machinery applies to both evaluation and training
- Offline evaluation (Level 3–4) → Interleaving (Level 4): Interleaving bypasses offline evaluation's fundamental limitations by using live data with within-user variance control

**The calibration thread:**
- You cannot trust an offline metric in isolation (Level 4)
- You must calibrate it against historical online experiments (Level 4)
- Industry companies have developed domain-specific calibrated metrics (Level 5)
- The calibration itself can be corrupted by Goodhart's Law (Level 2) if the calibration target is misspecified

---

# APPENDIX: QUICK REFERENCE

## A.1 Estimator Summary Table

| Estimator | Formula | Unbiased? | Variance | Key Assumption | When to Use |
|-----------|---------|-----------|----------|---------------|-------------|
| Direct Method (DM) | $\frac{1}{n}\sum_t \hat{r}(u_t, \pi_e(u_t))$ | Only if $\hat{r}$ is correct | Low | Reward model correct | Never in isolation |
| IPS | $\frac{1}{n}\sum_t \frac{\pi_e(\mathbf{a}_t)}{\pi_0(\mathbf{a}_t)} r_t$ | Yes (under overlap) | High | $\pi_e \ll \pi_0$, propensities correct | When propensities known, policies similar |
| SNIPS | Normalized IPS | No (biased $O(1/n)$) | Lower | Same as IPS | Large $n$, variance reduction needed |
| DR | DM + IPS residual | If either DM or IPS correct | Medium | One model correct | Default choice |
| Replay | Rejection sampling | Yes (under uniform $\pi_0$) | High | $\pi_0$ is uniform | Bandit evaluation with exploration |

## A.2 Metric Selection Guide

| Scenario | Recommended Metric | Avoid |
|----------|-------------------|-------|
| Single correct answer (navigational) | MRR, ERR | MAP (overkill) |
| Multiple relevant docs, binary relevance | MAP | NDCG (loses binary precision) |
| Graded relevance, cascade user | NDCG, ERR | Precision@K |
| User finds any result acceptable | HitRate@K | Recall@K |
| Catalog fairness important | Coverage + NDCG | NDCG alone |
| Long-session engagement | Diversity + Novelty + NDCG | CTR alone |
| Conversion/purchase | DR estimator, booking-adjusted NDCG | Click-based NDCG |

## A.3 Key Equations at a Glance

$$\text{NDCG@K} = \frac{\sum_{k=1}^{K} \frac{2^{\text{rel}_k}-1}{\log_2(k+1)}}{\text{IDCG@K}}$$

$$\text{ERR} = \sum_{k=1}^{K} \frac{1}{k} R_k \prod_{j=1}^{k-1}(1-R_j)$$

$$\hat{V}_{\text{IPS}}(\pi_e) = \frac{1}{n}\sum_{t=1}^{n} \frac{\pi_e(\mathbf{a}_t|u_t)}{\pi_0(\mathbf{a}_t|u_t)} r_t$$

$$\hat{V}_{\text{DR}}(\pi_e) = \frac{1}{n}\sum_t \mathbb{E}_{\pi_e}[\hat{r}(u_t, \mathbf{a})] + \frac{1}{n}\sum_t \frac{\pi_e(\mathbf{a}_t|u_t)}{\pi_0(\mathbf{a}_t|u_t)}(r_t - \hat{r}(u_t, \mathbf{a}_t))$$

$$P(\text{click}|i, k) = \theta_k \cdot \alpha_{u,i} \quad \text{(Position-Based Model)}$$

$$\text{ILD}(S_K) = \frac{2}{K(K-1)} \sum_{i \neq j, i,j \in S_K} d(i, j)$$

## A.4 Reading Order for Papers

If time is limited, read in this order:

1. **Schnabel et al. (2016)** — Establishes the causal framework. Read this first.
2. **Swaminathan & Joachims (2015)** — CRM provides the training-time application.
3. **Joachims et al. (2017)** — Applies IPS to LTR specifically.
4. **Wang et al. (2018)** — Propensity estimation in production.
5. **Saito & Joachims (2021)** — Extension to post-click signals.

For theory depth, add:
- Dudík et al. (2011) "Doubly Robust Policy Evaluation and Learning" (the DR foundations)
- Ai et al. (2018) "Unbiased LambdaMART" (DLA technical details)
- Chapelle et al. (2012) "Large-Scale Validation and Analysis of Interleaved Search Evaluation"

---

<a id='appendix-b'></a>
# APPENDIX B: RECOMMENDED READING — TEXTBOOKS AND ARTICLES

> A curated map of the literature organized by what each resource actually gives you. Grouped by type and ordered within each group by recommended reading priority.

---

## B.1 Textbooks

**"Introduction to Information Retrieval" — Manning, Raghavan & Schütze (2008)**
Free at [nlp.stanford.edu/IR-book](https://nlp.stanford.edu/IR-book/). Chapter 8 is the clearest formal derivation of Precision@K, Recall@K, MAP, NDCG, and interpolated precision-recall curves that exists in textbook form. The math is rigorous but accessible. This is the right starting point for metric foundations (Level 2).

**"Recommender Systems Handbook" — Ricci, Rokach & Shapira (eds.), 2nd ed. (2015)**
Springer. Two chapters are directly relevant:
- *"Evaluating Recommendation Systems"* by Shani & Gunawardana — the most comprehensive single treatment of offline evaluation methodology for recommenders. Covers accuracy metrics, beyond-accuracy metrics, experimental design, and statistical testing.
- *"Novelty and Diversity in Recommender Systems"* by Castells, Hurley & Vargas — formal treatment of the beyond-accuracy metrics (ILD, novelty, serendipity, coverage) that Level 2 only sketches.

**"Bandit Algorithms" — Lattimore & Szepesvári (2020)**
Free at [tor-lattimore.com/downloads/book/book.pdf](https://tor-lattimore.com/downloads/book/book.pdf). The theoretical foundation for why offline evaluation of recommendation policies is hard. Chapters on importance weighting, off-policy evaluation, and the exploration-exploitation tradeoff give the bandit-theoretic perspective that the ML papers in Level 5 assume without deriving. Mathematically demanding but worth it.

**"Causal Inference for Statistics, Social, and Biomedical Sciences" — Imbens & Rubin (2015)**
Cambridge. The gold standard reference for the potential outcomes framework (Rubin causal model) that Schnabel et al. (2016) applied to recommendations. Chapters on propensity scores, IPW estimation, and doubly robust estimators are the rigorous derivations behind everything in Level 3.

**"Trustworthy Online Controlled Experiments" — Kohavi, Tang & Xu (2020)**
Cambridge. Focused on A/B testing rather than offline evaluation, but essential for understanding the offline-online gap. Chapters on metric sensitivity, minimum detectable effects, novelty effects, and the taxonomy of why A/B tests fail are directly relevant to Level 4. Written by practitioners from Microsoft, Google, and LinkedIn.

---

## B.2 Survey Papers

**Shani & Gunawardana — "Evaluating Recommendation Systems" (2011)**
Chapter in the Recommender Systems Handbook, also available standalone. The most thorough survey of offline evaluation methodology for recommenders. Covers user studies, offline experiments, and online experiments with explicit discussion of their relative validity.

**Herlocker et al. — "Evaluating Collaborative Filtering Recommender Systems" (2004)**
ACM TOIS. The foundational paper that systematized how evaluation is done for collaborative filtering. Introduced many of the practices (train/test splits, holdout strategies, metric choices) that became standard — and later turned out to have systematic problems. Read it to understand what the field standardized on and why those choices are now contested.

**Voloshin et al. — "Empirical Study of Off-Policy Policy Evaluation for Reinforcement Learning" (NeurIPS 2021)**
Systematic empirical comparison of OPE estimators (DM, IPS, SNIPS, DR and variants) across domains. The findings transfer directly to recommendation evaluation: DR does not uniformly dominate, and the relative ranking of estimators depends heavily on the policy gap $\delta(\pi_0, \pi_e)$ and the quality of the reward model.

---

## B.3 Critical Papers That Changed How the Field Thinks About Evaluation

**Dacrema, Cremonesi & Jannach — "Are We Really Making Much Progress? A Worrying Analysis of Recent Neural Recommendation Approaches" (RecSys 2019)**
Must-read. Demonstrated that the majority of neural recommendation papers at top venues between 2015–2018 could not be replicated, and that properly tuned simple baselines (ItemKNN, BPR) matched or outperformed the proposed methods. The root cause was systematic evaluation methodology failures: inconsistent train/test splits, inconsistent negative sampling, and cherry-picked baselines. A direct indictment of the problems described in Level 1 and Level 2.

**Rendle et al. — "Revisiting the Performance of iALS on Item Recommendation Benchmarks" (2022)**
Follows up Dacrema et al. Shows that iALS (a 2008 algorithm) with careful tuning matches state-of-the-art neural methods on standard benchmarks. The evaluation methodology section is an excellent practical guide to what fair offline evaluation requires.

**Canamares & Castells — "Should I Follow the Crowd? A Probabilistic Analysis of the Effectiveness of Popularity in Recommender Systems" (SIGIR 2018)**
Proves analytically that under standard evaluation methodology, a pure-popularity recommender has artificially inflated metrics, and derives the exact conditions under which this occurs. Provides the formal complement to the popularity bias discussion in Level 1.

**Li et al. — "Unbiased Offline Evaluation of Contextual-Bandit-Based News Article Recommendation Algorithms" (WSDM 2011)**
The original Replay method paper. Clean derivation, explicit assumptions, and explicit discussion of when the method fails. Read alongside Level 3.5.

**Bottou et al. — "Counterfactual Reasoning and Learning Systems: The Example of Computational Advertising" (JMLR 2013)**
The paper that brought importance weighting and counterfactual evaluation into industrial ML. Predates Schnabel et al. and Joachims et al. and gives a broader systems-level view of why counterfactual reasoning is necessary at scale.

---

## B.4 Specialized Resources by Sub-Topic

**Interleaving:**
- Radlinski & Craswell, *"Optimized Interleaving for Online Retrieval Evaluation"* (WSDM 2013) — the definitive theory paper.
- Chapelle et al., *"Large-Scale Validation and Analysis of Interleaved Search Evaluation"* (ACM TOIS 2012) — empirical validation that interleaving results predict A/B test outcomes at Yahoo scale.

**Learning to Rank:**
- Liu, *"Learning to Rank for Information Retrieval"* (Foundations and Trends in IR, 2011) — free survey covering pointwise/pairwise/listwise LTR with evaluation implications.

**Off-Policy Evaluation (broader RL perspective):**
- Uehara et al., *"Review of Off-Policy Evaluation"* (2022) — comprehensive mathematical survey of the estimator landscape beyond the recommendation-specific literature.

---

## B.5 Recommended Reading Order

If time is limited, read in this order:

| Priority | Resource | Why |
|----------|----------|-----|
| 1 | Manning et al. Ch. 8 | Metric foundations — 2 hours |
| 2 | Shani & Gunawardana | Evaluation methodology overview — 3 hours |
| 3 | Dacrema et al. (2019) | Calibrate your skepticism — 1 hour |
| 4 | Schnabel et al. (2016) | Causal framework for evaluation |
| 5 | Joachims et al. (2017) | IPS applied to LTR |
| 6 | Imbens & Rubin Chs. 12–13 | Causal theory depth |
| 7 | Kohavi et al. book | Online eval side that motivates offline work |

---

<a id='appendix-c'></a>
# APPENDIX C: SEARCH VS. RECOMMENDATION — WHERE THE FRAMEWORK APPLIES

> The core mathematical framework (bandit feedback, IPS, propensity estimation, counterfactual evaluation) is domain-agnostic. But the practical emphasis shifts significantly between search and recommendation. This appendix maps those differences precisely.

---

## C.1 What the Two Settings Share

The fundamental problem is identical in both: you observe outcomes only under the logging policy, and you want to evaluate a new policy. Position bias, selection bias, the policy gap, IPS/SNIPS/DR estimators, and the offline-online calibration problem all apply with equal force to a search ranker and a recommendation system. The math does not care which domain you are in.

NDCG in particular is the dominant metric in **both** — it originated in IR/search (Järvelin & Kekäläinen, 2002) but is now standard in recommendation evaluation as well.

---

## C.2 Where Search Has Distinct Emphasis

**1. Explicit queries anchor relevance.**
Search has a query. Relevance can be defined at the (query, document) level rather than the (user, document) level. This makes it possible to collect **human relevance judgments** at scale — trained raters assess (query, document) pairs without needing to know who the user is. This is simply not possible for pure recommendations, where relevance is personalized.

**2. Cascade user models are native to search.**
ERR, the Position-Based Model, and the cascade model of user attention were all developed in the context of search result pages, where users scan a vertical list from top to bottom and stop when satisfied. Carousels, grids, and infinite feeds in recommendation systems have two-dimensional or non-linear attention patterns that cascade models poorly capture.

**3. The debiasing literature was born in search.**
Joachims et al. (2017), Wang et al. (2018), and the entire Unbiased LTR line of work were motivated by and evaluated on search click logs (MSLR, Yahoo LTR, Google personal search). Propensity estimation techniques — especially intervention harvesting from position swaps — are operationally natural in search because queries provide a stable unit of analysis.

**4. Benchmark datasets with near-complete relevance labels exist.**
MS MARCO, TREC, BEIR, and the Yahoo/Istella LTR datasets have human-labeled or near-complete relevance assessments. This makes it possible to compute MAP, NDCG, and MRR without worrying that your ground truth is biased by the logging policy — because the labels came from human judges, not from clicks.

---

## C.3 Where Recommendation Has Distinct Emphasis

**1. Popularity bias is the dominant structural problem.**
In search, the query filters what is relevant — a query for "transformer architecture paper" does not surface cat videos regardless of their popularity. In recommendation, there is no such anchor. Popularity is a strong confound: popular items have more training signal, get recommended more, get clicked more, and accumulate even more signal. The Zipfian feedback loop is much more severe in recommendation than in search.

**2. The MNAR problem is more severe.**
In search, you at least know what documents were retrieved for a given query (the retrieval set is defined by the query). In collaborative filtering, the set of items a user was ever exposed to is a tiny, non-random slice of the catalog, selected entirely by the system's estimate of their preferences. The missing-not-at-random structure is more deeply entangled with the system's own decisions.

**3. Beyond-accuracy metrics are first-class, not secondary.**
Coverage, ILD, novelty, and serendipity matter enormously in recommendation because there is no query to serve — the system is shaping what the user discovers. Filter bubbles, recommendation collapse, and long-tail supply fairness are structural concerns in recommendation that do not have clean analogues in query-driven search. In search, if you have a good answer to the query, diversity is secondary. In feed recommendation, diversity is a primary product requirement.

**4. Evaluation datasets almost always use implicit feedback with no ground truth.**
MovieLens, Amazon Reviews, Netflix Prize, LastFM — all use historical interactions as proxy labels. There are no human judges and no complete relevance assessments. Every metric computed on these datasets is a biased estimate of true preference. The Dacrema et al. (2019) critique about reproducibility applies with full force here.

**5. Leave-one-out evaluation is the dominant protocol — and it is flawed.**
Most recommendation papers hold out the last interaction per user and evaluate whether it appears in the top-K recommendations. This creates a single-relevant-item evaluation problem where HitRate@K and Recall@K collapse (Section 2.6), and the "ground truth" is a single click that may have been accidental or position-biased.

---

## C.4 Summary: Where to Focus Your Effort by Domain

| Dimension | Search | Recommendation |
|---|---|---|
| Dominant bias | Position bias | Popularity + selection bias |
| Ground truth source | Human relevance judges possible | Almost always implicit clicks |
| User model | Cascade (ERR, PBM) well-suited | Cascade poorly fits grids/feeds |
| Primary metrics | NDCG, MRR, MAP, ERR | NDCG, Recall@K, HitRate@K |
| Beyond-accuracy metrics | Secondary | First-class |
| Debiasing literature depth | Very deep (LTR is a 20-year field) | Growing but shallower |
| Academic benchmarks | Near-complete labels (TREC, MS MARCO) | Biased implicit feedback (MovieLens etc.) |
| Policy gap severity | Moderate (query constrains the space) | High (no query, wide open item space) |

**If you work on search:** The highest-leverage sections are Level 1 (position and selection bias), Level 3 (IPS and position-based debiasing), and Level 4 (CLTR and Unbiased LambdaMART). The debiasing techniques were built for your setting.

**If you work on recommendation:** The highest-leverage sections are Level 1 (MNAR and popularity bias framing), Level 2 (beyond-accuracy metrics and Goodhart's Law), and Level 4 (offline-online calibration, since the offline-online gap is generally worse in recommendation due to the deeper policy gap). Level 3 techniques still apply but propensity estimation is harder without an explicit query.

**If you work on personalized search** (Amazon product search, Airbnb, LinkedIn Jobs, Google personal search): you are in both simultaneously, and every part of the framework applies at full strength.